In [1]:
#Importacion de librerias
import requests
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
from unidecode import unidecode
import numpy as np
from tqdm import tqdm
import json
from pandas import json_normalize
import openpyxl
import concurrent.futures
from googleapiclient.discovery import build
from google.oauth2 import service_account


periodos = ['26ES5', '2026D', '26V05']
periodo_virtual = '26V05'
periodo_presencial = '2026D'
periodo_especializacion = '26ES5'
lider = 'LUZ'
periodo_pago = '202609'

In [2]:
#CREACION DE LA CONEXION CON BASE DE DATOS SQL SERVER

# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   #url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)
   url = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print('Error:',e)

Conexion a la base de datos realizada


In [3]:
# Autenticación con la clave de servicio
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly']
KEY = 'key2.json'  # LLAVES
SPREADSHEET_ID = "15NkOzO6MBlMJjtabMTB1Bc_PEKJ6lb6Rh8won1i1xm0"  # ID DE LA HOJA DE CALCULO

# Crear credenciales
creds = service_account.Credentials.from_service_account_file(KEY, scopes=SCOPES)
service = build('sheets', 'v4', credentials=creds)
sheet = service.spreadsheets()

# Llamar a la API de Google Sheets
RANGE_NAME = "'VENTAS'!A1:AE100000000"  # HOJA Y RANGO A TRABAJAR

result = sheet.values().get(spreadsheetId=SPREADSHEET_ID, range=RANGE_NAME).execute()
values = result.get('values', [])

# Mostrar resultados
if not values:
    print("No se encontraron datos.")
else:
    print(f"Datos obtenidos: {len(values):,} filas")

Datos obtenidos: 21,780 filas


In [4]:
df_excel = pd.DataFrame(values[1:], columns=values[0])
df_excel.shape

(21779, 31)

In [5]:
df_pre_ventas = df_excel[df_excel['Periodo'].isin(periodos)].copy()
df_pre_ventas

,Hora agregado,Dirección IP,Numero de recibo,Propietario del registro,Nombre de Estudiante,Ciudad de residencia,Número de Documento de estudiante,Tipo de inscripción,Programa vendido,Modalidad,...,% De Descuento,Valor pagado,Medio de pago,Fuente de pionero,Campana,Fuente campana,Content,Sitio web,ID único,Fecha
12553,05-16-2026 13:21:07,181.128.120.184,9738-1105460763-260514043236,Vinculaciones Margarita Rosa Navarro Garzon,capera varón Juan pablo,Colombia - Tolima - Ibague,1105460763,Normal,ADMINISTRACION PUBLICA,Virtual,...,60%,"$90.000,00",Pse,,,,,,13529,16/05/2026
13165,05-27-2026 10:38:12,190.184.201.12,9741-1118365041-260425115705,Vinculaciones Laura Yovana Bedoya Caballos,LAURA OLARTE LOPEZ,Colombia - Antioquia - Medellin,1118365041,Normal,ADMINISTRACION DE EMPRESAS,Presencial,...,60%,"$700.000,00",Banco,,,,,,14141,27/05/2026
15331,07-30-2026 12:43:33,191.156.159.249,9870-88242954-260722093908,Vinculaciones David Alejandro Orjuela Chaparro,HECTOR ALEXANDER HIGUERA DE LOS REYES,Colombia - Boyaca - Tunja,88242954,Normal,INGENIERIA INDUSTRIAL,Virtual,...,"0,1","$83.674,00",Pse,,,,,,16391,30/07/2026
15364,07-30-2026 15:00:02,38.210.96.76,9889-16499517-260728010742,Vinculaciones Miryam Daniela Callejas Perez,Carlos Hernán Valencia Hurtado,Colombia - Valle Del Cauca - Buenaventura,16499517,Normal,ESPECIALIZACIÓN EN GERENCIA EDUCATIVA,Virtual,...,"0,6","$193.948,00",Banco,,,,,,16424,30/07/2026
15468,07-31-2026 10:45:51,38.210.96.72,9895-1122141658-260729021731,Vinculaciones Miryam Daniela Callejas Perez,Luis Alberto Cordoba Gonzalez,Colombia - Meta - Villavicencio,1122141658,Normal,ESPECIALIZACION EN CONTRATACION ESTATAL,Virtual,...,"0,6","$193.948,00",Pse,,,,,,16528,31/07/2026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20857,10-06-2026 22:05:36,191.156.18.44,10026-1044218402-261004060249,Vinculaciones Jesus David Jimenez Amado,RAFAEL EDUARDO ALVEAR MEDINA,Colombia - Atlantico - Barranquilla,1044218402,Normal,NEGOCIOS INTERNACIONALES,Virtual,...,"0,6","$42.400,00",Pse,,,,,,20870,6/10/2026
20858,10-06-2026 22:23:19,191.156.18.44,9604-1066098610-260930061607,Vinculaciones Jesus David Jimenez Amado,Ingrid Gissela Carrero Carrascal,Colombia - Cesar - Curumani,1066098610,Normal,ESPECIALIZACION EN CONTRATACION ESTATAL,Virtual,...,"0,6","$154.360,00",Pse,,,,,,20871,6/10/2026
21775,,,,,,,,,,,...,None,None,None,None,None,None,None,None,None,None
21776,,,,,,,,,,,...,None,None,None,None,None,None,None,None,None,None


# SE COMPLEMENTA LA INFORMACION DE LOS ASESORES

In [6]:
host2_1 = '172.16.1.33'
port2_1 = '1433'
database2_1 = 'CUN_REPOSITORIO'

try:
    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_1 = create_engine(url2_1) 
    print("Conexion a la base de datos realizada")
    #archivo_orig = pd.read_sql("select * from CUN_REPOSITORIO.Comisiones.Com_Telec_Oct_2024_2_yCT", engine2_1)
    df_cedula = pd.read_sql(f'''
                    SELECT IDENTIFICACION AS 'Documento Ejecutivo', BOX_MAIL AS 'Correo Ejecutivo' FROM Hub_Empleados
                               ''', engine2_1)
    df_cedula_2 = df_cedula.copy()
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_1' in locals():
        engine2_1.dispose()
        print("Conexión cerrada")
df_cedula_2.shape

Conexion a la base de datos realizada
Conexión cerrada


(89771, 2)

In [7]:
df_cedula_2['Documento Ejecutivo'] = df_cedula_2['Documento Ejecutivo'].astype(int).astype(str)
df_cedula_2['Correo Ejecutivo'] = df_cedula_2['Correo Ejecutivo'].str.upper()
df_cedula_2

,Documento Ejecutivo,Correo Ejecutivo
0,28538359,DIANA_ACOSTAC@CUN.EDU.CO
1,1075234239,YENNY_DIAZ@CUN.EDU.CO
2,36953386,ASTRID_ERASO@CUN.EDU.CO
3,19385060,GERMAN_IBAGON@CUN.EDU.CO
4,1072366328,JOHAN_RODRIGUEZ@CUN.EDU.CO
...,...,...
89766,1074128157,YINER_UMANAG@CUN.EDU.CO
89767,1110583837,DANNA_PEREZ@CUN.EDU.CO
89768,1121838723,YAMILE_RODRIGUEZ@CUN.EDU.CO
89769,1141318847,SARA_PASTRANAR@CUN.EDU.CO


In [8]:
df_pre_ventas['Correo Ejecutivo'] = df_pre_ventas['Correo Ejecutivo'].str.upper()

In [9]:
# se unifica con la informacion de registro ventas
df_pre_ventas = df_pre_ventas.merge(df_cedula_2,how='inner',left_on='Correo Ejecutivo',right_on='Correo Ejecutivo').copy()
df_pre_ventas.drop_duplicates(inplace=True)
df_pre_ventas

,Hora agregado,Dirección IP,Numero de recibo,Propietario del registro,Nombre de Estudiante,Ciudad de residencia,Número de Documento de estudiante,Tipo de inscripción,Programa vendido,Modalidad,...,Valor pagado,Medio de pago,Fuente de pionero,Campana,Fuente campana,Content,Sitio web,ID único,Fecha,Documento Ejecutivo_y
0,05-16-2026 13:21:07,181.128.120.184,9738-1105460763-260514043236,Vinculaciones Margarita Rosa Navarro Garzon,capera varón Juan pablo,Colombia - Tolima - Ibague,1105460763,Normal,ADMINISTRACION PUBLICA,Virtual,...,"$90.000,00",Pse,,,,,,13529,16/05/2026,1110587566
49,05-27-2026 10:38:12,190.184.201.12,9741-1118365041-260425115705,Vinculaciones Laura Yovana Bedoya Caballos,LAURA OLARTE LOPEZ,Colombia - Antioquia - Medellin,1118365041,Normal,ADMINISTRACION DE EMPRESAS,Presencial,...,"$700.000,00",Banco,,,,,,14141,27/05/2026,1033377195
58,07-30-2026 12:43:33,191.156.159.249,9870-88242954-260722093908,Vinculaciones David Alejandro Orjuela Chaparro,HECTOR ALEXANDER HIGUERA DE LOS REYES,Colombia - Boyaca - Tunja,88242954,Normal,INGENIERIA INDUSTRIAL,Virtual,...,"$83.674,00",Pse,,,,,,16391,30/07/2026,1025521082
65,07-30-2026 15:00:02,38.210.96.76,9889-16499517-260728010742,Vinculaciones Miryam Daniela Callejas Perez,Carlos Hernán Valencia Hurtado,Colombia - Valle Del Cauca - Buenaventura,16499517,Normal,ESPECIALIZACIÓN EN GERENCIA EDUCATIVA,Virtual,...,"$193.948,00",Banco,,,,,,16424,30/07/2026,1053616791
72,07-31-2026 10:45:51,38.210.96.72,9895-1122141658-260729021731,Vinculaciones Miryam Daniela Callejas Perez,Luis Alberto Cordoba Gonzalez,Colombia - Meta - Villavicencio,1122141658,Normal,ESPECIALIZACION EN CONTRATACION ESTATAL,Virtual,...,"$193.948,00",Pse,,,,,,16528,31/07/2026,1053616791
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56521,10-06-2026 22:05:36,191.156.18.44,10026-1044218402-261004060249,Vinculaciones Jesus David Jimenez Amado,RAFAEL EDUARDO ALVEAR MEDINA,Colombia - Atlantico - Barranquilla,1044218402,Normal,NEGOCIOS INTERNACIONALES,Virtual,...,"$42.400,00",Pse,,,,,,20870,6/10/2026,1081811110
56530,10-06-2026 22:23:19,191.156.18.44,9604-1066098610-260930061607,Vinculaciones Jesus David Jimenez Amado,Ingrid Gissela Carrero Carrascal,Colombia - Cesar - Curumani,1066098610,Normal,ESPECIALIZACION EN CONTRATACION ESTATAL,Virtual,...,"$154.360,00",Pse,,,,,,20871,6/10/2026,1081811110
56539,,,,,,,,,,,...,None,None,None,None,None,None,None,None,None,1019099501
56540,,,,,,,,,,,...,None,None,None,None,None,None,None,None,None,1019099501


# METAS POR PERIODO Y POR LIDER

In [10]:
df_metas_virtual = pd.read_excel(r'C:\Users\alejandro_arboledas\OneDrive - Corporación Unificada Nacional de Educación Superior - CUN\Documentos\PYTHON\VINCULACIONES\MODELO3\2026\METAS_BOGOTA_202609.xlsx', sheet_name= periodo_virtual)
df_metas_virtual

,GESTOR DOCUMENTO,GESTOR CORREO,CATEGORIA,Meta,LIDER,SEDE,LOCALIDAD
0,1053814476,paula_lopezco@cun.edu.co,BRONCE,20,LUZ,BOGOTA,SOACHA
1,1012421425,diego_gonzalezar@cun.edu.co,BRONCE,20,LUZ,BOGOTA,SOACHA
2,1030576351,katherine_acosta@cun.edu.co,BRONCE,20,LUZ,BOGOTA,SOACHA
3,1014249751,jose_espitiaz@cun.edu.co,BRONCE,20,LUZ,BOGOTA,SOACHA
4,1018504409,maria_pradac@cun.edu.co,DIAMANTE,40,LUZ,BOGOTA,RAFAEL URIBE URIBE
5,1003068849,paula_vence@cun.edu.co,DIAMANTE,40,LUZ,BOGOTA,RAFAEL URIBE URIBE
6,52957207,joyce_mora@cun.edu.co,PLATA,30,LUZ,BOGOTA,RAFAEL URIBE URIBE
7,1007101612,daniel_prieto@cun.edu.co,DIAMANTE,40,LUZ,BOGOTA,RAFAEL URIBE URIBE
8,53154239,diana_sabogalpa@cun.edu.co,BRONCE,20,LUZ,BOGOTA,CIUDAD BOLIVAR
9,1013680182,sara_castanedae@cun.edu.co,PLATA,30,LUZ,BOGOTA,CIUDAD BOLIVAR


In [11]:
df_metas_virtual = df_metas_virtual[df_metas_virtual['LIDER'] == lider]
df_metas_virtual.shape

(46, 7)

In [12]:
df_metas_virtual['GESTOR CORREO'] = df_metas_virtual['GESTOR CORREO'].str.upper()
df_ventas_virtual = df_pre_ventas[df_pre_ventas['Correo Ejecutivo'].isin(df_metas_virtual['GESTOR CORREO'])].copy()
df_ventas_virtual = df_ventas_virtual[df_ventas_virtual['Periodo'].isin([periodo_virtual])]
df_ventas_virtual.shape

(1002, 32)

# SE DEJAN SOLO LAS COLUMNAS NECESARIAS


In [13]:
columnas = ['Hora agregado', 'Numero de recibo','Dirección IP','Correo del estudiante','Teléfono del estudiante',
'Ciudad de residencia','Imagen del Pago Generado','% De Descuento',
'Medio de pago','Fuente de pionero','Campana','Fuente campana','Content','Sitio web', 'Tipo de inscripción',
'ID único','Fecha','Documento Ejecutivo_x','Número de aprobación de pago','Tipo de Financiación', 'Convenio', 'Valor pagado']

df_ventas_virtual.drop(columns=columnas, inplace=True)
df_ventas_virtual['Periodo'] = df_ventas_virtual['Periodo'].astype(str).str.strip()
df_ventas_virtual = df_ventas_virtual.rename(columns={'Documento Ejecutivo_y':'Documento Ejecutivo'})
df_ventas_virtual['Documento Ejecutivo'] = df_ventas_virtual['Documento Ejecutivo'].astype(str)
df_ventas_virtual['Número de Documento de estudiante'] = df_ventas_virtual['Número de Documento de estudiante'].astype(str)
df_ventas_virtual['Llave']=df_ventas_virtual['Número de Documento de estudiante']+df_ventas_virtual['Periodo']
df_ventas_virtual.shape


(1002, 11)

# SE TRAEN LA MARCA DE PAGO Y LA FUERZA COMERCIAL

In [14]:
periodos_sql = ",".join(f"'{p}'" for p in periodos)

In [15]:
para_orden = df_pre_ventas['Número de Documento de estudiante'].unique()
valores_unicos_df1_orden = "', '".join(map(str, para_orden))
valores_unicos_df1_orden = f"'{valores_unicos_df1_orden}'"

In [16]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(f''' 
        SELECT distinct DOC_ALUM,PERIODO,ESTADO_PAGO,fuerzazoho as FuerzaComercialFinal, NUEVO, CLASE_ACTUAL, TIPO, PROGRAMA, EST_MATRICULADO
        FROM ZOHO.Base_Personas WITH(NOLOCK)
        WHERE PERIODO IN ({periodos_sql})
        and doc_alum in ({valores_unicos_df1_orden})
                        ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_recibos = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()
df_recibos.shape

(3976, 9)

In [17]:
df_recibos['LLAVE_RECIBOS'] = df_recibos['DOC_ALUM'].astype(str)+df_recibos['PERIODO']
df_recibos_virtual = df_recibos[df_recibos['PERIODO'] == periodo_virtual].copy()
df_recibos_virtual.shape

(1528, 10)

# SE CRUZA LOS RECIBOS CON EL REGISTRO DE VENTAS

In [18]:
df_recibos_virtual = df_recibos_virtual.merge(df_ventas_virtual[['Propietario del registro','Modalidad','Correo Ejecutivo', 'Regional',
       'Documento Ejecutivo', 'Llave']],how='inner',left_on='LLAVE_RECIBOS',right_on='Llave')
df_recibos_virtual.drop_duplicates(inplace=True)
df_recibos_virtual.shape

(996, 16)

In [19]:
#tratamiento de columnas
df_recibos_virtual['CLASE_ACTUAL'] = df_recibos_virtual['CLASE_ACTUAL'].astype(str).str.upper()
df_recibos_virtual['FuerzaComercialFinal'] = df_recibos_virtual['FuerzaComercialFinal'].str.upper()
df_recibos_virtual['NUEVO'] = df_recibos_virtual['NUEVO'].astype(str).str.upper()
df_recibos_virtual['ESTADO_PAGO'] = df_recibos_virtual['ESTADO_PAGO'].astype(str).str.upper()
df_recibos_virtual['FuerzaComercialFinal'] = df_recibos_virtual['FuerzaComercialFinal'].str.strip()
df_recibos_virtual['NUEVO'] = df_recibos_virtual['NUEVO'].str.strip()
df_recibos_virtual['CLASE_ACTUAL'] = df_recibos_virtual['CLASE_ACTUAL'].str.strip()
df_recibos_virtual['ESTADO_PAGO'] = df_recibos_virtual['ESTADO_PAGO'].str.strip()
df_recibos_virtual['EST_MATRICULADO'] = (
    df_recibos_virtual['EST_MATRICULADO']
    .astype(str)
    .str.strip()
    .str.upper()
)

In [20]:
# se divide entre validos
df_validos_virtual = df_recibos_virtual[
    ((df_recibos_virtual['FuerzaComercialFinal'] == 'BASICA Y MEDIA')  | (df_recibos_virtual['FuerzaComercialFinal'] == 'VINCULACIONES') | (df_recibos_virtual['FuerzaComercialFinal'] == 'INDETERMINADO')) &
    (df_recibos_virtual['NUEVO'] == 'NUEVO') &
    (df_recibos_virtual['EST_MATRICULADO'] == '1-ACTIVO') & 
    (df_recibos_virtual['ESTADO_PAGO'] == 'PAGO')
].copy()
df_validos_virtual.drop_duplicates(inplace=True)
df_validos_virtual.shape

(947, 16)

In [21]:
# se divide entre no validos
mask = (
    df_recibos_virtual['FuerzaComercialFinal'].isin(
        ['BASICA Y MEDIA', 'VINCULACIONES', 'INDETERMINADO']
    ) &
    (df_recibos_virtual['NUEVO'] == 'NUEVO') &
    (df_recibos_virtual['EST_MATRICULADO'] == '1-ACTIVO') &
    (df_recibos_virtual['ESTADO_PAGO'] == 'PAGO')
)
df_no_validos_virtual = df_recibos_virtual[~mask].copy()
df_no_validos_virtual.drop_duplicates(inplace=True)
df_no_validos_virtual.shape

(49, 16)

In [22]:
# Inicializar MOTIVO vacío
df_no_validos_virtual['MOTIVO'] = ''

# Condición 1
condicion = df_no_validos_virtual['ESTADO_PAGO'] == 'NO PAGO'
df_no_validos_virtual.loc[condicion, 'MOTIVO'] += 'NO REGISTRA PAGO | '

# Condición 2
condicion = (
    (df_no_validos_virtual['NUEVO'] != 'NUEVO')
)
df_no_validos_virtual.loc[condicion, 'MOTIVO'] += 'NO ES UN ESTUDIANTE NUEVO | '

# Condición 3
condicion = df_no_validos_virtual['FuerzaComercialFinal'] != 'VINCULACIONES'
df_no_validos_virtual.loc[condicion, 'MOTIVO'] += 'PERTENECE A OTRA FUERZA | '

# Condición 4
condicion = df_no_validos_virtual['EST_MATRICULADO'] != '1-ACTIVO'
df_no_validos_virtual.loc[condicion, 'MOTIVO'] += 'MATRICULA INACTIVA | '

# Quitar el último " | "
df_no_validos_virtual['MOTIVO'] = (
    df_no_validos_virtual['MOTIVO']
    .str.rstrip(' |')
)

# Los que no tengan ningún motivo quedan como N/A
df_no_validos_virtual.loc[
    df_no_validos_virtual['MOTIVO'] == '',
    'MOTIVO'
] = 'N/A'

df_no_validos_virtual

,DOC_ALUM,PERIODO,ESTADO_PAGO,FuerzaComercialFinal,NUEVO,CLASE_ACTUAL,TIPO,PROGRAMA,EST_MATRICULADO,LLAVE_RECIBOS,Propietario del registro,Modalidad,Correo Ejecutivo,Regional,Documento Ejecutivo,Llave,MOTIVO
6,1000217592,26V05,PAGO,CONTACT,NUEVO,NUEVO,Pregrado,DIRECCIÓN Y PRODUCCIÓN DE MEDIOS AUDIOVISUALES,1-ACTIVO,100021759226V05,Vinculaciones Laura Camila Castillo Sepulveda,Presencial,LAURA_CASTILLOSEP@CUN.EDU.CO,Bogotá,1125230726,100021759226V05,PERTENECE A OTRA FUERZA
10,1000148866,26V05,PAGO,TELECAMPUS,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS CONTABLES,1-ACTIVO,100014886626V05,Vinculaciones Paula Alejandra Lopez Correa,Virtual,PAULA_LOPEZCO@CUN.EDU.CO,Bogotá,1053814476,100014886626V05,PERTENECE A OTRA FUERZA
18,1000351487,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS OPERATIVOS DE ...,1-ACTIVO,100035148726V05,Vinculaciones Erika Andrea Ramirez Rico,Virtual,ERIKA_RAMIREZ@CUN.EDU.CO,Bogotá,1012382480,100035148726V05,NO REGISTRA PAGO
78,1000789899,26V05,PAGO,PROYECTOS ESPECIALES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN REGISTRO DE PROCESOS PR...,1-ACTIVO,100078989926V05,Vinculaciones Erika Andrea Arias Molano,Virtual,ERIKA_ARIASM@CUN.EDU.CO,Bogotá,1024525114,100078989926V05,PERTENECE A OTRA FUERZA
138,1007811840,26V05,PAGO,CONTACT,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS ADMINISTRATIVOS,1-ACTIVO,100781184026V05,Vinculaciones Cristian Hernando Pulido Aguilar,Virtual,CRISTIAN_PULIDOA@CUN.EDU.CO,Bogotá,1033736941,100781184026V05,PERTENECE A OTRA FUERZA
143,1010174559,26V05,PAGO,CONTACT,NUEVO,NUEVO,Pregrado,CONTADURÍA PÚBLICA,1-ACTIVO,101017455926V05,Vinculaciones Daniel Felipe Prieto Alvarado,Virtual,DANIEL_PRIETO@CUN.EDU.CO,Bogotá,1007101612,101017455926V05,PERTENECE A OTRA FUERZA
160,1007160052,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN SOPORTE DE SISTEMAS E I...,1-ACTIVO,100716005226V05,Vinculaciones Erika Andrea Ramirez Rico,Virtual,ERIKA_RAMIREZ@CUN.EDU.CO,Bogotá,1012382480,100716005226V05,NO REGISTRA PAGO
183,1012370878,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN SOPORTE DE SISTEMAS E I...,1-ACTIVO,101237087826V05,Vinculaciones Katherine Acosta Rojas,Virtual,KATHERINE_ACOSTA@CUN.EDU.CO,Bogotá,1030576351,101237087826V05,NO REGISTRA PAGO
191,1013117576,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN SOPORTE DE SISTEMAS E I...,1-ACTIVO,101311757626V05,Vinculaciones Sara Valentina Castañeda Espitia,Virtual,SARA_CASTANEDAE@CUN.EDU.CO,Bogotá,1014187111,101311757626V05,NO REGISTRA PAGO
195,1016066198,26V05,PAGO,CONTACT,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS ADMINISTRATIVOS,1-ACTIVO,101606619826V05,Vinculaciones Paula Nataly Quiroz Morales,Virtual,PAULA_QUIROZ@CUN.EDU.CO,Bogotá,1016035848,101606619826V05,PERTENECE A OTRA FUERZA


In [23]:
# se traen las ventas que no tienen marca de pago
df_no_recibos_virtual = df_ventas_virtual[~df_ventas_virtual['Llave'].isin(df_recibos_virtual['LLAVE_RECIBOS'])].copy()
df_no_recibos_virtual['DOC_ALUM'] = df_no_recibos_virtual['Número de Documento de estudiante'].astype(str)
df_no_recibos_virtual['PERIODO'] = df_no_recibos_virtual['Periodo']
df_no_recibos_virtual['ESTADO_PAGO'] = ' '
df_no_recibos_virtual['EST_MATRICULADO'] = ' '
df_no_recibos_virtual['FuerzaComercialFinal'] = ' '
df_no_recibos_virtual['NUEVO'] = ' '
df_no_recibos_virtual['CLASE_ACTUAL'] = ' '
df_no_recibos_virtual['TIPO'] = ' '
df_no_recibos_virtual['PROGRAMA'] = df_no_recibos_virtual['Programa vendido'] 
df_no_recibos_virtual['LLAVE_RECIBOS'] = ' '
df_no_recibos_virtual['MOTIVO'] = 'NO CUENTA CON MARCA DE PAGO'
df_no_recibos_virtual = df_no_recibos_virtual[['DOC_ALUM', 'PERIODO', 'ESTADO_PAGO', 'FuerzaComercialFinal', 'NUEVO',
       'CLASE_ACTUAL', 'TIPO', 'PROGRAMA', 'EST_MATRICULADO', 'LLAVE_RECIBOS',
       'Propietario del registro', 'Modalidad', 'Correo Ejecutivo', 'Regional',
       'Documento Ejecutivo', 'Llave','MOTIVO']]
df_no_recibos_virtual.shape

(23, 17)

In [24]:
# SE UNIFICAN LOS NO VALIDOS Y LOS QUE NO TIENEN MARCA DE PAGO
df_no_validos_2 = pd.concat([df_no_validos_virtual,df_no_recibos_virtual])
df_no_validos_2

,DOC_ALUM,PERIODO,ESTADO_PAGO,FuerzaComercialFinal,NUEVO,CLASE_ACTUAL,TIPO,PROGRAMA,EST_MATRICULADO,LLAVE_RECIBOS,Propietario del registro,Modalidad,Correo Ejecutivo,Regional,Documento Ejecutivo,Llave,MOTIVO
6,1000217592,26V05,PAGO,CONTACT,NUEVO,NUEVO,Pregrado,DIRECCIÓN Y PRODUCCIÓN DE MEDIOS AUDIOVISUALES,1-ACTIVO,100021759226V05,Vinculaciones Laura Camila Castillo Sepulveda,Presencial,LAURA_CASTILLOSEP@CUN.EDU.CO,Bogotá,1125230726,100021759226V05,PERTENECE A OTRA FUERZA
10,1000148866,26V05,PAGO,TELECAMPUS,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS CONTABLES,1-ACTIVO,100014886626V05,Vinculaciones Paula Alejandra Lopez Correa,Virtual,PAULA_LOPEZCO@CUN.EDU.CO,Bogotá,1053814476,100014886626V05,PERTENECE A OTRA FUERZA
18,1000351487,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS OPERATIVOS DE ...,1-ACTIVO,100035148726V05,Vinculaciones Erika Andrea Ramirez Rico,Virtual,ERIKA_RAMIREZ@CUN.EDU.CO,Bogotá,1012382480,100035148726V05,NO REGISTRA PAGO
78,1000789899,26V05,PAGO,PROYECTOS ESPECIALES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN REGISTRO DE PROCESOS PR...,1-ACTIVO,100078989926V05,Vinculaciones Erika Andrea Arias Molano,Virtual,ERIKA_ARIASM@CUN.EDU.CO,Bogotá,1024525114,100078989926V05,PERTENECE A OTRA FUERZA
138,1007811840,26V05,PAGO,CONTACT,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS ADMINISTRATIVOS,1-ACTIVO,100781184026V05,Vinculaciones Cristian Hernando Pulido Aguilar,Virtual,CRISTIAN_PULIDOA@CUN.EDU.CO,Bogotá,1033736941,100781184026V05,PERTENECE A OTRA FUERZA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43649,1121922283,26V05,,,,,,ESPECIALIZACION EN PAZ Y DESARROLLO TERRITORIAL,,,Vinculaciones Natalia Remolina Herrera,Virtual,NATALIA_REMOLINAH@CUN.EDU.CO,Bogotá,1026592868,112192228326V05,NO CUENTA CON MARCA DE PAGO
48418,1121929905,26V05,,,,,,ESPECIALIZACIÓN EN GERENCIA DE PROYECTOS,,,Vinculaciones Yolima Toloza Gonzalez,Virtual,YOLIMA_TOLOZA@CUN.EDU.CO,Bogotá,35393797,112192990526V05,NO CUENTA CON MARCA DE PAGO
50909,1022997317,26V05,,,,,,ESPECIALIZACIÓN EN GERENCIA EDUCATIVA,,,Vinculaciones Diana Clemencia Sabogal Parrado,Virtual,DIANA_SABOGALPA@CUN.EDU.CO,Bogotá,53154239,102299731726V05,NO CUENTA CON MARCA DE PAGO
51476,79975882,26V05,,,,,,ESPECIALIZACIÓN EN GERENCIA FINANCIERA,,,Vinculaciones Diana Clemencia Sabogal Parrado,Virtual,DIANA_SABOGALPA@CUN.EDU.CO,Bogotá,53154239,7997588226V05,NO CUENTA CON MARCA DE PAGO


# SE TRAEN LAS OPORTUNIDADES DE ZOHO CRM

In [25]:
host2_1 = '172.16.1.33'
port2_1 = '1433'
database2_1 = 'CUN_REPOSITORIO'

try:
    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_1 = create_engine(url2_1) 
    print("Conexion a la base de datos realizada")
    df_zoho = pd.read_sql(f'''
Select distinct 
o.Propietario_de_Oportunidad_Name,
o.Hora_de_creación,
o.Fecha_de_cierre,
o.Fase,
o.Hora_de_modificación,
o.Número_de_Documento,
o.Programa_de_interés,
o.PERIODO,
o.Hora_de_la_última_actividad,
u.Correo_electrónico
from zoho.CRM.Oportunidades o
left join zoho.CRM.Usuarios U
ON o.Propietario_de_Oportunidad = u.Id
    where Periodo in ({periodos_sql})
and Fase LIKE '%Matriculado%'
                               ''', engine2_1)
    Consulta_z2 = df_zoho.copy()
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_1' in locals():
        engine2_1.dispose()
        print("Conexión cerrada")
df_zoho.shape

Conexion a la base de datos realizada
Conexión cerrada


(14831, 10)

In [26]:
df_zoho['Correo_electrónico'] = df_zoho['Correo_electrónico'].str.upper()
df_zoho['LLAVE_ZOHO']=df_zoho['Número_de_Documento'].astype(str)+df_zoho['PERIODO'].astype(str).str.strip()+df_zoho['Correo_electrónico']
df_zoho_vinculaciones = df_zoho[df_zoho['Correo_electrónico'].isin(df_pre_ventas['Correo Ejecutivo'])].copy()
df_zoho_vinculaciones = df_zoho_vinculaciones.drop_duplicates()
df_zoho_vinculaciones.shape

(3725, 11)

In [27]:
# se crea la llave para cruzar por doc alum+periodo+correo ejecutivo
df_validos_virtual['LLAVE_ZOHO'] = df_validos_virtual['DOC_ALUM']+df_validos_virtual['PERIODO']+df_validos_virtual['Correo Ejecutivo']

In [28]:
df_zoho_recibos_virtual = df_validos_virtual.merge(df_zoho_vinculaciones[['Propietario_de_Oportunidad_Name','Fase','Hora_de_modificación','LLAVE_ZOHO']],left_on='LLAVE_ZOHO',right_on='LLAVE_ZOHO',how='inner').copy()
df_zoho_recibos_virtual.drop_duplicates(inplace=True)
df_zoho_recibos_virtual.shape

(833, 20)

In [29]:
df_no_validos_zoho = df_validos_virtual[~df_validos_virtual['LLAVE_ZOHO'].isin(df_zoho_vinculaciones['LLAVE_ZOHO'])].copy()
df_no_validos_zoho['MOTIVO'] = 'NO ESTA OK EN ZOHO'
df_no_validos_zoho = df_no_validos_zoho.drop(columns=['LLAVE_ZOHO'])
df_no_validos_zoho.shape

(115, 17)

In [30]:
# SE UNIFICAN LOS NO VALIDOS TOTALES DE VIRTUALES
df_no_validos_3 = pd.concat([df_no_validos_2,df_no_validos_zoho])
df_no_validos_3

,DOC_ALUM,PERIODO,ESTADO_PAGO,FuerzaComercialFinal,NUEVO,CLASE_ACTUAL,TIPO,PROGRAMA,EST_MATRICULADO,LLAVE_RECIBOS,Propietario del registro,Modalidad,Correo Ejecutivo,Regional,Documento Ejecutivo,Llave,MOTIVO
6,1000217592,26V05,PAGO,CONTACT,NUEVO,NUEVO,Pregrado,DIRECCIÓN Y PRODUCCIÓN DE MEDIOS AUDIOVISUALES,1-ACTIVO,100021759226V05,Vinculaciones Laura Camila Castillo Sepulveda,Presencial,LAURA_CASTILLOSEP@CUN.EDU.CO,Bogotá,1125230726,100021759226V05,PERTENECE A OTRA FUERZA
10,1000148866,26V05,PAGO,TELECAMPUS,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS CONTABLES,1-ACTIVO,100014886626V05,Vinculaciones Paula Alejandra Lopez Correa,Virtual,PAULA_LOPEZCO@CUN.EDU.CO,Bogotá,1053814476,100014886626V05,PERTENECE A OTRA FUERZA
18,1000351487,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS OPERATIVOS DE ...,1-ACTIVO,100035148726V05,Vinculaciones Erika Andrea Ramirez Rico,Virtual,ERIKA_RAMIREZ@CUN.EDU.CO,Bogotá,1012382480,100035148726V05,NO REGISTRA PAGO
78,1000789899,26V05,PAGO,PROYECTOS ESPECIALES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN REGISTRO DE PROCESOS PR...,1-ACTIVO,100078989926V05,Vinculaciones Erika Andrea Arias Molano,Virtual,ERIKA_ARIASM@CUN.EDU.CO,Bogotá,1024525114,100078989926V05,PERTENECE A OTRA FUERZA
138,1007811840,26V05,PAGO,CONTACT,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS ADMINISTRATIVOS,1-ACTIVO,100781184026V05,Vinculaciones Cristian Hernando Pulido Aguilar,Virtual,CRISTIAN_PULIDOA@CUN.EDU.CO,Bogotá,1033736941,100781184026V05,PERTENECE A OTRA FUERZA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
957,1121910809,26V05,PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PRODUCCIÓN DE MATERIAS ...,1-ACTIVO,112191080926V05,Vinculaciones Yolima Toloza Gonzalez,Virtual,YOLIMA_TOLOZA@CUN.EDU.CO,Bogotá,35393797,112191080926V05,NO ESTA OK EN ZOHO
976,20500031,26V05,PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS COMUNICATIVOS ...,1-ACTIVO,2050003126V05,Vinculaciones Erika Andrea Ramirez Rico,Virtual,ERIKA_RAMIREZ@CUN.EDU.CO,Bogotá,1012382480,2050003126V05,NO ESTA OK EN ZOHO
981,42159636,26V05,PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN REGISTRO DE PROCESOS PR...,1-ACTIVO,4215963626V05,Vinculaciones Yolima Toloza Gonzalez,Virtual,YOLIMA_TOLOZA@CUN.EDU.CO,Bogotá,35393797,4215963626V05,NO ESTA OK EN ZOHO
992,79219176,26V05,PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN SOPORTE DE SISTEMAS E I...,1-ACTIVO,7921917626V05,Vinculaciones Katherine Acosta Rojas,Virtual,KATHERINE_ACOSTA@CUN.EDU.CO,Bogotá,1030576351,7921917626V05,NO ESTA OK EN ZOHO


# SE CRUZA CON EL HISTORICO 

In [31]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 
            select * from CUN_REPOSITORIO.comisiones.COMISIONES_VINCULACIONES_ACTUALES
            WHERE [COMISION TOTAL] != 0
 ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_historico = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()
df_historico.shape

(3407, 33)

In [32]:
df_historico['LLAVE_HISTORICO'] = df_historico['Documento_Estudiante'].astype(str) + df_historico['PERIODO'].astype(str)

# SE DEJA TODOS LOS VALIDOS EN UN SOLO DATAFRAME

In [33]:
df_validos_virtuales_final = df_zoho_recibos_virtual[~df_zoho_recibos_virtual['Llave'].isin(df_historico['LLAVE_HISTORICO'])].copy()
df_validos_virtuales_final.shape

(833, 20)

# SE DEJA TODOS LOS NO VALIDOS VIRTUALES EN UN SOLO DATAFRAME

In [34]:
df_no_validos_historico = df_zoho_recibos_virtual[df_zoho_recibos_virtual['Llave'].isin(df_historico['LLAVE_HISTORICO'])].copy()
df_no_validos_historico['MOTIVO'] = 'YA SE PAGO EN LA LIQUIDACION PASADA' 
df_no_validos_historico = df_no_validos_historico.drop(columns=['LLAVE_ZOHO','Propietario_de_Oportunidad_Name','Fase','Hora_de_modificación'])
df_no_validos_historico.shape

(0, 17)

In [35]:
df_novalidos_virtual_final = pd.concat([df_no_validos_3,df_no_validos_historico])
df_novalidos_virtual_final.shape

(187, 17)

# CALCULOS

In [36]:
# Filtrar y contar las ventas por estado de pago (PAGO y NO PAGO)
ventas_asesor_virtuales = (
    df_validos_virtuales_final
    .groupby(["Documento Ejecutivo"])
    .size()
    .reset_index(name="VENTAS_VIRTUALES")
).copy()


In [37]:
df_metas_virtual['GESTOR DOCUMENTO'] = df_metas_virtual['GESTOR DOCUMENTO'].astype(str)
ventas_asesor_virtuales['Documento Ejecutivo'] = ventas_asesor_virtuales['Documento Ejecutivo'].astype(str)
ventas_asesor_virtuales = ventas_asesor_virtuales.merge(df_metas_virtual,how='left',left_on='Documento Ejecutivo',right_on='GESTOR DOCUMENTO').copy()
ventas_asesor_virtuales = ventas_asesor_virtuales.drop(columns=['GESTOR DOCUMENTO'])
ventas_asesor_virtuales

,Documento Ejecutivo,VENTAS_VIRTUALES,GESTOR CORREO,CATEGORIA,Meta,LIDER,SEDE,LOCALIDAD
0,1000335802,6,ANGIE_DIAZJ@CUN.EDU.CO,ESMERALDA,50.0,LUZ,BOGOTA,GRANDES CUENTAS
1,1000457601,14,DIEGO_BOHORQUEZCO@CUN.EDU.CO,PLATA,30.0,LUZ,BOGOTA,ENGATIVA
2,1001056960,5,NaN,NaN,NaN,NaN,NaN,NaN
3,1001956960,5,PAULA_HURTADOM@CUN.EDU.CO,BRONCE,20.0,LUZ,BOGOTA,KENNEDY
4,1003068849,30,PAULA_VENCE@CUN.EDU.CO,DIAMANTE,40.0,LUZ,BOGOTA,RAFAEL URIBE URIBE
5,1007101612,44,DANIEL_PRIETO@CUN.EDU.CO,DIAMANTE,40.0,LUZ,BOGOTA,RAFAEL URIBE URIBE
6,1007702093,10,ANGGY_AGRAYVI@CUN.EDU.CO,HIERRO,10.0,LUZ,BOGOTA,SABANA OCCIDENTE
7,1007812062,1,JUAN_CASTANEDAA@CUN.EDU.CO,ESMERALDA,50.0,LUZ,BOGOTA,GRANDES CUENTAS
8,1012388460,55,DAIRON_TELLEZZA@CUN.EDU.CO,PLATA,30.0,LUZ,BOGOTA,BOSA
9,1012421425,24,DIEGO_GONZALEZAR@CUN.EDU.CO,BRONCE,20.0,LUZ,BOGOTA,SOACHA


# SE DEFINE EL ESQUEMA COMISIONAL

In [38]:
esquema_comisiones_virtuales = {
    "LUZ": {
        "ORO": {
                        60: [  
                                (60, float('inf'), 80000)
                            ]},

        "ESMERALDA": {
                        50: [  
                                (50, 59, 70000),
                                (60, float('inf'), 80000)
                            ]},
        "DIAMANTE": {
                        40: [  
                                (40, 49, 60000),
                                (50, 59, 70000),
                                (60, float('inf'), 80000)
                            ]},
        "PLATA": {
            
                        30: [  
                                (30, 39, 50000),    
                                (40, 49, 60000),
                                (50, 59, 70000),
                                (60, float('inf'), 80000)
                            ]},
        "BRONCE": {
            
                        20: [  
                                (20, 29, 40000),
                                (30, 39, 50000),    
                                (40, 49, 60000),
                                (50, 59, 70000),
                                (60, float('inf'), 80000)
                            ]},
        "HIERRO": {
            
                        10: [  
                                (10, 29, 40000),    
                                (30, 39, 50000),
                                (40, 49, 60000),
                                (50, 59, 70000),
                                (60, float('inf'), 80000)
                            ]}                               
    }}

In [39]:
def asignar_comision_virtuales(lider, categoria, meta, ventas):

    # 1️⃣ Validar líder
    if lider not in esquema_comisiones_virtuales:
        return 0

    # 2️⃣ Validar sede
    if categoria not in esquema_comisiones_virtuales[lider]:
        return 0

    # 4️⃣ Validar meta
    if meta not in esquema_comisiones_virtuales[lider][categoria]:
        return 0

    # 5️⃣ Validar cumplimiento de meta
    if ventas < meta:
        return 0

    # 6️⃣ Obtener los pisos correspondientes a la meta
    rangos = esquema_comisiones_virtuales[lider][categoria][meta]

    # 7️⃣ Buscar el piso correspondiente según las ventas
    for min_rango, max_rango, comision in rangos:

        if min_rango <= ventas <= max_rango:

            # Comisión por venta × número de ventas
            return ventas * comision

    return 0

In [40]:
ventas_asesor_virtuales["COMISION_TOTAL_VIRTUAL"] = ventas_asesor_virtuales.apply(
    lambda row: asignar_comision_virtuales(
        row["LIDER"],
        row["CATEGORIA"],
        row["Meta"],
        row["VENTAS_VIRTUALES"]
    ),
    axis=1
)
ventas_asesor_virtuales['COMISION_UNITARIA_VIRTUAL'] = ventas_asesor_virtuales['COMISION_TOTAL_VIRTUAL'] / ventas_asesor_virtuales['VENTAS_VIRTUALES']
ventas_asesor_virtuales['%CUMPLIMIENTO_VIRTUAL'] = (
    (ventas_asesor_virtuales['VENTAS_VIRTUALES'] / ventas_asesor_virtuales['Meta']) * 100).round(2).astype(str) + '%'
ventas_asesor_virtuales

,Documento Ejecutivo,VENTAS_VIRTUALES,GESTOR CORREO,CATEGORIA,Meta,LIDER,SEDE,LOCALIDAD,COMISION_TOTAL_VIRTUAL,COMISION_UNITARIA_VIRTUAL,%CUMPLIMIENTO_VIRTUAL
0,1000335802,6,ANGIE_DIAZJ@CUN.EDU.CO,ESMERALDA,50.0,LUZ,BOGOTA,GRANDES CUENTAS,0,0.0,12.0%
1,1000457601,14,DIEGO_BOHORQUEZCO@CUN.EDU.CO,PLATA,30.0,LUZ,BOGOTA,ENGATIVA,0,0.0,46.67%
2,1001056960,5,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,nan%
3,1001956960,5,PAULA_HURTADOM@CUN.EDU.CO,BRONCE,20.0,LUZ,BOGOTA,KENNEDY,0,0.0,25.0%
4,1003068849,30,PAULA_VENCE@CUN.EDU.CO,DIAMANTE,40.0,LUZ,BOGOTA,RAFAEL URIBE URIBE,0,0.0,75.0%
5,1007101612,44,DANIEL_PRIETO@CUN.EDU.CO,DIAMANTE,40.0,LUZ,BOGOTA,RAFAEL URIBE URIBE,2640000,60000.0,110.0%
6,1007702093,10,ANGGY_AGRAYVI@CUN.EDU.CO,HIERRO,10.0,LUZ,BOGOTA,SABANA OCCIDENTE,400000,40000.0,100.0%
7,1007812062,1,JUAN_CASTANEDAA@CUN.EDU.CO,ESMERALDA,50.0,LUZ,BOGOTA,GRANDES CUENTAS,0,0.0,2.0%
8,1012388460,55,DAIRON_TELLEZZA@CUN.EDU.CO,PLATA,30.0,LUZ,BOGOTA,BOSA,3850000,70000.0,183.33%
9,1012421425,24,DIEGO_GONZALEZAR@CUN.EDU.CO,BRONCE,20.0,LUZ,BOGOTA,SOACHA,960000,40000.0,120.0%


## PROCESO DE COMISION PARA PRESENCIALES
-- SE DEFINEN LAS METAS

In [41]:
df_metas_presencial = pd.read_excel(r'C:\Users\alejandro_arboledas\OneDrive - Corporación Unificada Nacional de Educación Superior - CUN\Documentos\PYTHON\VINCULACIONES\MODELO3\2026\METAS_BOGOTA_202609.xlsx', sheet_name= periodo_presencial)
df_metas_presencial

,GESTOR DOCUMENTO,GESTOR CORREO,CATEGORIA,Meta,LIDER,SEDE,LOCALIDAD
0,1053814476,paula_lopezco@cun.edu.co,BRONCE,5,LUZ,BOGOTA,SOACHA
1,1012421425,diego_gonzalezar@cun.edu.co,BRONCE,5,LUZ,BOGOTA,SOACHA
2,1030576351,katherine_acosta@cun.edu.co,BRONCE,5,LUZ,BOGOTA,SOACHA
3,1014249751,jose_espitiaz@cun.edu.co,BRONCE,5,LUZ,BOGOTA,SOACHA
4,1018504409,maria_pradac@cun.edu.co,DIAMANTE,15,LUZ,BOGOTA,RAFAEL URIBE URIBE
5,1003068849,paula_vence@cun.edu.co,DIAMANTE,15,LUZ,BOGOTA,RAFAEL URIBE URIBE
6,52957207,joyce_mora@cun.edu.co,PLATA,10,LUZ,BOGOTA,RAFAEL URIBE URIBE
7,1007101612,daniel_prieto@cun.edu.co,DIAMANTE,15,LUZ,BOGOTA,RAFAEL URIBE URIBE
8,53154239,diana_sabogalpa@cun.edu.co,BRONCE,5,LUZ,BOGOTA,CIUDAD BOLIVAR
9,1013680182,sara_castanedae@cun.edu.co,PLATA,10,LUZ,BOGOTA,CIUDAD BOLIVAR


In [42]:
df_metas_presencial = df_metas_presencial[df_metas_presencial['LIDER'] == lider]
df_metas_presencial.shape

(46, 7)

VENTAS PRESENCIALES

In [43]:
df_metas_presencial['GESTOR CORREO'] = df_metas_presencial['GESTOR CORREO'].str.upper()
df_ventas_presencial = df_pre_ventas[df_pre_ventas['Correo Ejecutivo'].isin(df_metas_presencial['GESTOR CORREO'])].copy()
df_ventas_presencial = df_ventas_presencial[df_ventas_presencial['Periodo'].isin([periodo_presencial])]
df_ventas_presencial.shape

(127, 32)

In [44]:
columnas = ['Hora agregado', 'Numero de recibo','Dirección IP','Correo del estudiante','Teléfono del estudiante',
'Ciudad de residencia','Imagen del Pago Generado','% De Descuento',
'Medio de pago','Fuente de pionero','Campana','Fuente campana','Content','Sitio web', 'Tipo de inscripción',
'ID único','Fecha','Documento Ejecutivo_x','Número de aprobación de pago','Tipo de Financiación', 'Convenio', 'Valor pagado']

df_ventas_presencial.drop(columns=columnas, inplace=True)
df_ventas_presencial['Periodo'] = df_ventas_presencial['Periodo'].astype(str).str.strip()
df_ventas_presencial = df_ventas_presencial.rename(columns={'Documento Ejecutivo_y':'Documento Ejecutivo'})
df_ventas_presencial['Documento Ejecutivo'] = df_ventas_presencial['Documento Ejecutivo'].astype(str)
df_ventas_presencial['Número de Documento de estudiante'] = df_ventas_presencial['Número de Documento de estudiante'].astype(str)
df_ventas_presencial['Llave']=df_ventas_presencial['Número de Documento de estudiante']+df_ventas_presencial['Periodo']
df_ventas_presencial.shape

(127, 11)

SE CRUZA CON RECIBOS

In [45]:
df_recibos_presencial = df_recibos[df_recibos['PERIODO'] == periodo_presencial].copy()
df_recibos_presencial.shape

(344, 10)

In [46]:
df_recibos_presencial = df_recibos_presencial.merge(df_ventas_presencial[['Propietario del registro','Modalidad','Correo Ejecutivo', 'Regional',
       'Documento Ejecutivo', 'Llave']],how='inner',left_on='LLAVE_RECIBOS',right_on='Llave')
df_recibos_presencial.drop_duplicates(inplace=True)
df_recibos_presencial.shape

(110, 16)

In [47]:
#tratamiento de columnas
df_recibos_presencial['CLASE_ACTUAL'] = df_recibos_presencial['CLASE_ACTUAL'].astype(str).str.upper()
df_recibos_presencial['FuerzaComercialFinal'] = df_recibos_presencial['FuerzaComercialFinal'].str.upper()
df_recibos_presencial['NUEVO'] = df_recibos_presencial['NUEVO'].astype(str).str.upper()
df_recibos_presencial['ESTADO_PAGO'] = df_recibos_presencial['ESTADO_PAGO'].astype(str).str.upper()
df_recibos_presencial['FuerzaComercialFinal'] = df_recibos_presencial['FuerzaComercialFinal'].str.strip()
df_recibos_presencial['NUEVO'] = df_recibos_presencial['NUEVO'].str.strip()
df_recibos_presencial['CLASE_ACTUAL'] = df_recibos_presencial['CLASE_ACTUAL'].str.strip()
df_recibos_presencial['ESTADO_PAGO'] = df_recibos_presencial['ESTADO_PAGO'].str.strip()
df_recibos_presencial['EST_MATRICULADO'] = (
    df_recibos_presencial['EST_MATRICULADO']
    .astype(str)
    .str.strip()
    .str.upper()
)

In [48]:
# se divide entre validos
df_validos_presencial = df_recibos_presencial[
    ((df_recibos_presencial['FuerzaComercialFinal'] == 'BASICA Y MEDIA') | (df_recibos_presencial['FuerzaComercialFinal'] == 'VINCULACIONES') | (df_recibos_presencial['FuerzaComercialFinal'] == 'INDETERMINADO')) &
    (df_recibos_presencial['NUEVO'] == 'NUEVO') &
    (df_recibos_presencial['EST_MATRICULADO'] == '1-ACTIVO') & 
    (df_recibos_presencial['ESTADO_PAGO'] == 'PAGO')
].copy()
df_validos_presencial.drop_duplicates(inplace=True)
df_validos_presencial.shape 

(105, 16)

In [49]:
# se divide entre no validos
mask = (
    df_recibos_presencial['FuerzaComercialFinal'].isin(
        ['BASICA Y MEDIA', 'VINCULACIONES', 'INDETERMINADO']
    ) &
    (df_recibos_presencial['NUEVO'] == 'NUEVO') &
    (df_recibos_presencial['EST_MATRICULADO'] == '1-ACTIVO') &
    (df_recibos_presencial['ESTADO_PAGO'] == 'PAGO')
)
df_no_validos_presencial = df_recibos_presencial[~mask].copy()
df_no_validos_presencial.drop_duplicates(inplace=True)
df_no_validos_presencial.shape

(5, 16)

In [50]:
# Inicializar MOTIVO vacío
df_no_validos_presencial['MOTIVO'] = ''

# Condición 1
condicion = df_no_validos_presencial['ESTADO_PAGO'] == 'NO PAGO'
df_no_validos_presencial.loc[condicion, 'MOTIVO'] += 'NO REGISTRA PAGO | '

# Condición 2
condicion = (
    (df_no_validos_presencial['NUEVO'] != 'NUEVO')
)
df_no_validos_presencial.loc[condicion, 'MOTIVO'] += 'NO ES UN ESTUDIANTE NUEVO | '

# Condición 3
condicion = df_no_validos_presencial['FuerzaComercialFinal'] != 'VINCULACIONES'
df_no_validos_presencial.loc[condicion, 'MOTIVO'] += 'PERTENECE A OTRA FUERZA | '

# Condición 4
condicion = df_no_validos_presencial['EST_MATRICULADO'] != '1-ACTIVO'
df_no_validos_presencial.loc[condicion, 'MOTIVO'] += 'MATRICULA INACTIVA | '

# Quitar el último " | "
df_no_validos_presencial['MOTIVO'] = (
    df_no_validos_presencial['MOTIVO']
    .str.rstrip(' |')
)

# Los que no tengan ningún motivo quedan como N/A
df_no_validos_presencial.loc[
    df_no_validos_presencial['MOTIVO'] == '',
    'MOTIVO'
] = 'N/A'

df_no_validos_presencial

,DOC_ALUM,PERIODO,ESTADO_PAGO,FuerzaComercialFinal,NUEVO,CLASE_ACTUAL,TIPO,PROGRAMA,EST_MATRICULADO,LLAVE_RECIBOS,Propietario del registro,Modalidad,Correo Ejecutivo,Regional,Documento Ejecutivo,Llave,MOTIVO
44,1021394206,2026D,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN MANTENIMIENTO ELECTRÓNICO,1-ACTIVO,10213942062026D,Vinculaciones Camilo Ernesto Leon Garzon,Presencial,CAMILO_LEONG@CUN.EDU.CO,Bogotá,1014237175,10213942062026D,NO REGISTRA PAGO
55,1028780631,2026D,PAGO,PERMANENCIA,ANTIGUO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS ADMINISTRATIVOS,1-ACTIVO,10287806312026D,Vinculaciones Katherine Acosta Rojas,Presencial,KATHERINE_ACOSTA@CUN.EDU.CO,Bogotá,1030576351,10287806312026D,NO ES UN ESTUDIANTE NUEVO | PERTENECE A OTRA F...
65,1050955076,2026D,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN SERVICIOS ADMINISTRATIV...,1-ACTIVO,10509550762026D,Vinculaciones Yudi Esperanza Barrero Romero,Presencial,YUDI_BARREROR@CUN.EDU.CO,Bogotá,1023868232,10509550762026D,NO REGISTRA PAGO
73,1033711094,2026D,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS OPERATIVOS DEL...,1-ACTIVO,10337110942026D,Vinculaciones Erika Andrea Ramirez Rico,Presencial,ERIKA_RAMIREZ@CUN.EDU.CO,Bogotá,1012382480,10337110942026D,NO REGISTRA PAGO
99,1033713105,2026D,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS OPERATIVOS DEL...,1-ACTIVO,10337131052026D,Vinculaciones Erika Andrea Ramirez Rico,Presencial,ERIKA_RAMIREZ@CUN.EDU.CO,Bogotá,1012382480,10337131052026D,NO REGISTRA PAGO


In [51]:
# se traen las ventas que no tienen marca de pago
df_no_recibos_presencial = df_ventas_presencial[~df_ventas_presencial['Llave'].isin(df_recibos_presencial['LLAVE_RECIBOS'])].copy()
df_no_recibos_presencial['DOC_ALUM'] = df_no_recibos_presencial['Número de Documento de estudiante'].astype(str)
df_no_recibos_presencial['PERIODO'] = df_no_recibos_presencial['Periodo']
df_no_recibos_presencial['ESTADO_PAGO'] = ' '
df_no_recibos_presencial['EST_MATRICULADO'] = ' '
df_no_recibos_presencial['FuerzaComercialFinal'] = ' '
df_no_recibos_presencial['NUEVO'] = ' '
df_no_recibos_presencial['CLASE_ACTUAL'] = ' '
df_no_recibos_presencial['TIPO'] = ' '
df_no_recibos_presencial['PROGRAMA'] = df_no_recibos_presencial['Programa vendido'] 
df_no_recibos_presencial['LLAVE_RECIBOS'] = ' '
df_no_recibos_presencial['MOTIVO'] = 'NO CUENTA CON MARCA DE PAGO'
df_no_recibos_presencial = df_no_recibos_presencial[['DOC_ALUM', 'PERIODO', 'ESTADO_PAGO', 'FuerzaComercialFinal', 'NUEVO',
       'CLASE_ACTUAL', 'TIPO', 'PROGRAMA', 'EST_MATRICULADO', 'LLAVE_RECIBOS',
       'Propietario del registro', 'Modalidad', 'Correo Ejecutivo', 'Regional',
       'Documento Ejecutivo', 'Llave','MOTIVO']]
df_no_recibos_presencial.shape

(17, 17)

In [52]:
# SE UNIFICAN LOS NO VALIDOS Y LOS QUE NO TIENEN MARCA DE PAGO
df_no_validos_presencial_2 = pd.concat([df_no_validos_presencial,df_no_recibos_presencial])
df_no_validos_presencial_2

,DOC_ALUM,PERIODO,ESTADO_PAGO,FuerzaComercialFinal,NUEVO,CLASE_ACTUAL,TIPO,PROGRAMA,EST_MATRICULADO,LLAVE_RECIBOS,Propietario del registro,Modalidad,Correo Ejecutivo,Regional,Documento Ejecutivo,Llave,MOTIVO
44,1021394206,2026D,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN MANTENIMIENTO ELECTRÓNICO,1-ACTIVO,10213942062026D,Vinculaciones Camilo Ernesto Leon Garzon,Presencial,CAMILO_LEONG@CUN.EDU.CO,Bogotá,1014237175,10213942062026D,NO REGISTRA PAGO
55,1028780631,2026D,PAGO,PERMANENCIA,ANTIGUO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS ADMINISTRATIVOS,1-ACTIVO,10287806312026D,Vinculaciones Katherine Acosta Rojas,Presencial,KATHERINE_ACOSTA@CUN.EDU.CO,Bogotá,1030576351,10287806312026D,NO ES UN ESTUDIANTE NUEVO | PERTENECE A OTRA F...
65,1050955076,2026D,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN SERVICIOS ADMINISTRATIV...,1-ACTIVO,10509550762026D,Vinculaciones Yudi Esperanza Barrero Romero,Presencial,YUDI_BARREROR@CUN.EDU.CO,Bogotá,1023868232,10509550762026D,NO REGISTRA PAGO
73,1033711094,2026D,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS OPERATIVOS DEL...,1-ACTIVO,10337110942026D,Vinculaciones Erika Andrea Ramirez Rico,Presencial,ERIKA_RAMIREZ@CUN.EDU.CO,Bogotá,1012382480,10337110942026D,NO REGISTRA PAGO
99,1033713105,2026D,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS OPERATIVOS DEL...,1-ACTIVO,10337131052026D,Vinculaciones Erika Andrea Ramirez Rico,Presencial,ERIKA_RAMIREZ@CUN.EDU.CO,Bogotá,1012382480,10337131052026D,NO REGISTRA PAGO
412,1034292129,2026D,,,,,,ADMINISTRACION DE EMPRESAS,,,Vinculaciones Paula Alejandra Lopez Correa,Presencial,PAULA_LOPEZCO@CUN.EDU.CO,Bogotá,1053814476,10342921292026D,NO CUENTA CON MARCA DE PAGO
1802,1024517410,2026D,,,,,,ADMINISTRACION DEPORTIVA,,,Vinculaciones Carolina Rocio Viloria Pedreros,Presencial,CAROLINA_VILORIAP@CUN.EDU.CO,Bogotá,1015393814,10245174102026D,NO CUENTA CON MARCA DE PAGO
3046,1013599906,2026D,,,,,,DISENO GRAFICO,,,Vinculaciones Dairon Felipe Tellez Zambrano,Presencial,DAIRON_TELLEZZA@CUN.EDU.CO,Bogotá,1012388460,10135999062026D,NO CUENTA CON MARCA DE PAGO
3080,1073672721,2026D,,,,,,DISENO DE MODAS,,,Vinculaciones Dairon Felipe Tellez Zambrano,Presencial,DAIRON_TELLEZZA@CUN.EDU.CO,Bogotá,1012388460,10736727212026D,NO CUENTA CON MARCA DE PAGO
7046,1069744707,2026D,,,,,,ADMINISTRACION DE EMPRESAS,,,Vinculaciones Dairon Felipe Tellez Zambrano,Presencial,DAIRON_TELLEZZA@CUN.EDU.CO,Bogotá,1012388460,10697447072026D,NO CUENTA CON MARCA DE PAGO


SE CRUZA CON OPORTUNIDADES

In [53]:
# se crea la llave para cruzar por doc alum+periodo+correo ejecutivo
df_validos_presencial['LLAVE_ZOHO'] = df_validos_presencial['DOC_ALUM']+df_validos_presencial['PERIODO']+df_validos_presencial['Correo Ejecutivo']

In [54]:
df_zoho_recibos_presencial = df_validos_presencial.merge(df_zoho_vinculaciones[['Propietario_de_Oportunidad_Name','Fase','Hora_de_modificación','LLAVE_ZOHO']],left_on='LLAVE_ZOHO',right_on='LLAVE_ZOHO',how='inner').copy()
df_zoho_recibos_presencial.drop_duplicates(inplace=True)
df_zoho_recibos_presencial.shape

(90, 20)

In [55]:
df_no_validos_zoho_presencial = df_validos_presencial[~df_validos_presencial['LLAVE_ZOHO'].isin(df_zoho_vinculaciones['LLAVE_ZOHO'])].copy()
df_no_validos_zoho_presencial['MOTIVO'] = 'NO ESTA OK EN ZOHO'
df_no_validos_zoho_presencial = df_no_validos_zoho_presencial.drop(columns=['LLAVE_ZOHO'])
df_no_validos_zoho_presencial.shape

(15, 17)

In [56]:
# SE UNIFICAN LOS NO VALIDOS TOTALES DE VIRTUALES
df_no_validos_presencial_3 = pd.concat([df_no_validos_presencial_2,df_no_validos_zoho_presencial])
df_no_validos_presencial_3

,DOC_ALUM,PERIODO,ESTADO_PAGO,FuerzaComercialFinal,NUEVO,CLASE_ACTUAL,TIPO,PROGRAMA,EST_MATRICULADO,LLAVE_RECIBOS,Propietario del registro,Modalidad,Correo Ejecutivo,Regional,Documento Ejecutivo,Llave,MOTIVO
44,1021394206,2026D,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN MANTENIMIENTO ELECTRÓNICO,1-ACTIVO,10213942062026D,Vinculaciones Camilo Ernesto Leon Garzon,Presencial,CAMILO_LEONG@CUN.EDU.CO,Bogotá,1014237175,10213942062026D,NO REGISTRA PAGO
55,1028780631,2026D,PAGO,PERMANENCIA,ANTIGUO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS ADMINISTRATIVOS,1-ACTIVO,10287806312026D,Vinculaciones Katherine Acosta Rojas,Presencial,KATHERINE_ACOSTA@CUN.EDU.CO,Bogotá,1030576351,10287806312026D,NO ES UN ESTUDIANTE NUEVO | PERTENECE A OTRA F...
65,1050955076,2026D,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN SERVICIOS ADMINISTRATIV...,1-ACTIVO,10509550762026D,Vinculaciones Yudi Esperanza Barrero Romero,Presencial,YUDI_BARREROR@CUN.EDU.CO,Bogotá,1023868232,10509550762026D,NO REGISTRA PAGO
73,1033711094,2026D,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS OPERATIVOS DEL...,1-ACTIVO,10337110942026D,Vinculaciones Erika Andrea Ramirez Rico,Presencial,ERIKA_RAMIREZ@CUN.EDU.CO,Bogotá,1012382480,10337110942026D,NO REGISTRA PAGO
99,1033713105,2026D,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS OPERATIVOS DEL...,1-ACTIVO,10337131052026D,Vinculaciones Erika Andrea Ramirez Rico,Presencial,ERIKA_RAMIREZ@CUN.EDU.CO,Bogotá,1012382480,10337131052026D,NO REGISTRA PAGO
412,1034292129,2026D,,,,,,ADMINISTRACION DE EMPRESAS,,,Vinculaciones Paula Alejandra Lopez Correa,Presencial,PAULA_LOPEZCO@CUN.EDU.CO,Bogotá,1053814476,10342921292026D,NO CUENTA CON MARCA DE PAGO
1802,1024517410,2026D,,,,,,ADMINISTRACION DEPORTIVA,,,Vinculaciones Carolina Rocio Viloria Pedreros,Presencial,CAROLINA_VILORIAP@CUN.EDU.CO,Bogotá,1015393814,10245174102026D,NO CUENTA CON MARCA DE PAGO
3046,1013599906,2026D,,,,,,DISENO GRAFICO,,,Vinculaciones Dairon Felipe Tellez Zambrano,Presencial,DAIRON_TELLEZZA@CUN.EDU.CO,Bogotá,1012388460,10135999062026D,NO CUENTA CON MARCA DE PAGO
3080,1073672721,2026D,,,,,,DISENO DE MODAS,,,Vinculaciones Dairon Felipe Tellez Zambrano,Presencial,DAIRON_TELLEZZA@CUN.EDU.CO,Bogotá,1012388460,10736727212026D,NO CUENTA CON MARCA DE PAGO
7046,1069744707,2026D,,,,,,ADMINISTRACION DE EMPRESAS,,,Vinculaciones Dairon Felipe Tellez Zambrano,Presencial,DAIRON_TELLEZZA@CUN.EDU.CO,Bogotá,1012388460,10697447072026D,NO CUENTA CON MARCA DE PAGO


SE CRUZA CON EL HISTORICO

In [57]:
df_validos_presenciales_final = df_zoho_recibos_presencial[~df_zoho_recibos_presencial['Llave'].isin(df_historico['LLAVE_HISTORICO'])].copy()
df_validos_presenciales_final.shape

(90, 20)

In [58]:
df_no_validos_historico_presenciales = df_zoho_recibos_presencial[df_zoho_recibos_presencial['Llave'].isin(df_historico['LLAVE_HISTORICO'])].copy()
df_no_validos_historico_presenciales['MOTIVO'] = 'YA SE PAGO EN LA LIQUIDACION PASADA' 
df_no_validos_historico_presenciales = df_no_validos_historico_presenciales.drop(columns=['LLAVE_ZOHO','Propietario_de_Oportunidad_Name','Fase','Hora_de_modificación'])
df_no_validos_historico_presenciales.shape

(0, 17)

SE UNE TODOS LOS NO VALIDOS PRESENCIAL

In [59]:
df_novalidos_presencial_final = pd.concat([df_no_validos_presencial_3,df_no_validos_historico_presenciales])
df_novalidos_presencial_final.shape

(37, 17)

CALCULOS

In [60]:
# Filtrar y contar las ventas por estado de pago (PAGO y NO PAGO)
ventas_asesor_presenciales = (
    df_validos_presenciales_final
    .groupby(["Documento Ejecutivo"])
    .size()
    .reset_index(name="VENTAS_PRESENCIALES")
).copy()

In [61]:
df_metas_presencial['GESTOR DOCUMENTO'] = df_metas_presencial['GESTOR DOCUMENTO'].astype(str)
ventas_asesor_presenciales['Documento Ejecutivo'] = ventas_asesor_presenciales['Documento Ejecutivo'].astype(str)
ventas_asesor_presenciales = ventas_asesor_presenciales.merge(df_metas_presencial,how='left',left_on='Documento Ejecutivo',right_on='GESTOR DOCUMENTO').copy()
ventas_asesor_presenciales = ventas_asesor_presenciales.drop(columns=['GESTOR DOCUMENTO'])
ventas_asesor_presenciales

,Documento Ejecutivo,VENTAS_PRESENCIALES,GESTOR CORREO,CATEGORIA,Meta,LIDER,SEDE,LOCALIDAD
0,1000335802,3,ANGIE_DIAZJ@CUN.EDU.CO,ESMERALDA,20.0,LUZ,BOGOTA,GRANDES CUENTAS
1,1001056960,2,NaN,NaN,NaN,NaN,NaN,NaN
2,1001956960,2,PAULA_HURTADOM@CUN.EDU.CO,BRONCE,5.0,LUZ,BOGOTA,KENNEDY
3,1003068849,3,PAULA_VENCE@CUN.EDU.CO,DIAMANTE,15.0,LUZ,BOGOTA,RAFAEL URIBE URIBE
4,1007101612,9,DANIEL_PRIETO@CUN.EDU.CO,DIAMANTE,15.0,LUZ,BOGOTA,RAFAEL URIBE URIBE
5,1012388460,1,DAIRON_TELLEZZA@CUN.EDU.CO,PLATA,10.0,LUZ,BOGOTA,BOSA
6,1014187111,11,NaN,NaN,NaN,NaN,NaN,NaN
7,1014237175,4,CAMILO_LEONG@CUN.EDU.CO,BRONCE,5.0,LUZ,BOGOTA,ENGATIVA
8,1014249751,2,JOSE_ESPITIAZ@CUN.EDU.CO,BRONCE,5.0,LUZ,BOGOTA,SOACHA
9,1014260333,6,CHRISTIAN_FAGUAHE@CUN.EDU.CO,PLATA,10.0,LUZ,BOGOTA,SUBA


In [62]:
esquema_comisiones_presenciales = {

    "LUZ": {
        "ORO": {
                        25: [  
                                (25, float('inf'), 80000)
                            ]},

        "ESMERALDA": {
                        20: [  
                                (20, 24, 70000),
                                (25, float('inf'), 80000)
                            ]},
        "DIAMANTE": {
                        15: [  
                                (15, 19, 60000),
                                (20, 24, 70000),
                                (25, float('inf'), 80000)
                            ]},
        "PLATA": {
            
                        10: [  
                                (10, 14, 50000),    
                                (15, 19, 60000),
                                (20, 24, 70000),
                                (25, float('inf'), 80000)
                            ]},
        "BRONCE": {
            
                        5: [  
                                (5, 9, 40000),
                                (10, 14, 50000),    
                                (15, 19, 60000),
                                (20, 24, 70000),
                                (25, float('inf'), 80000)
                            ]}           
    }}

In [63]:
def asignar_comision_presenciales(lider, categoria, meta, ventas):

    # 1️⃣ Validar líder
    if lider not in esquema_comisiones_presenciales:
        return 0

    # 2️⃣ Validar sede
    if categoria not in esquema_comisiones_presenciales[lider]:
        return 0
    # 4️⃣ Validar meta
    if meta not in esquema_comisiones_presenciales[lider][categoria]:
        return 0
    # 5️⃣ Validar cumplimiento de meta
    if ventas < meta:
        return 0

    # 6️⃣ Obtener los pisos correspondientes a la meta
    rangos = esquema_comisiones_presenciales[lider][categoria][meta]

    # 7️⃣ Buscar el piso correspondiente según las ventas
    for min_rango, max_rango, comision in rangos:

        if min_rango <= ventas <= max_rango:

            # Comisión por venta × número de ventas
            return ventas * comision

    return 0

In [64]:
ventas_asesor_presenciales["COMISION_TOTAL_PRESENCIAL"] = ventas_asesor_presenciales.apply(
    lambda row: asignar_comision_presenciales(
        row["LIDER"],
        row["CATEGORIA"],
        row["Meta"],
        row["VENTAS_PRESENCIALES"]
    ),
    axis=1
)
ventas_asesor_presenciales['COMISION_UNITARIA_PRESENCIAL'] = ventas_asesor_presenciales['COMISION_TOTAL_PRESENCIAL'] / ventas_asesor_presenciales['VENTAS_PRESENCIALES']
ventas_asesor_presenciales['%CUMPLIMIENTO_PRESENCIAL'] = (
    (ventas_asesor_presenciales['VENTAS_PRESENCIALES'] / ventas_asesor_presenciales['Meta']) * 100).round(2).astype(str) + '%'
ventas_asesor_presenciales

,Documento Ejecutivo,VENTAS_PRESENCIALES,GESTOR CORREO,CATEGORIA,Meta,LIDER,SEDE,LOCALIDAD,COMISION_TOTAL_PRESENCIAL,COMISION_UNITARIA_PRESENCIAL,%CUMPLIMIENTO_PRESENCIAL
0,1000335802,3,ANGIE_DIAZJ@CUN.EDU.CO,ESMERALDA,20.0,LUZ,BOGOTA,GRANDES CUENTAS,0,0.0,15.0%
1,1001056960,2,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,nan%
2,1001956960,2,PAULA_HURTADOM@CUN.EDU.CO,BRONCE,5.0,LUZ,BOGOTA,KENNEDY,0,0.0,40.0%
3,1003068849,3,PAULA_VENCE@CUN.EDU.CO,DIAMANTE,15.0,LUZ,BOGOTA,RAFAEL URIBE URIBE,0,0.0,20.0%
4,1007101612,9,DANIEL_PRIETO@CUN.EDU.CO,DIAMANTE,15.0,LUZ,BOGOTA,RAFAEL URIBE URIBE,0,0.0,60.0%
5,1012388460,1,DAIRON_TELLEZZA@CUN.EDU.CO,PLATA,10.0,LUZ,BOGOTA,BOSA,0,0.0,10.0%
6,1014187111,11,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,nan%
7,1014237175,4,CAMILO_LEONG@CUN.EDU.CO,BRONCE,5.0,LUZ,BOGOTA,ENGATIVA,0,0.0,80.0%
8,1014249751,2,JOSE_ESPITIAZ@CUN.EDU.CO,BRONCE,5.0,LUZ,BOGOTA,SOACHA,0,0.0,40.0%
9,1014260333,6,CHRISTIAN_FAGUAHE@CUN.EDU.CO,PLATA,10.0,LUZ,BOGOTA,SUBA,0,0.0,60.0%


# SE LIQUIDAN ESPECIALIZACIONES

In [65]:
df_metas_especializacion = pd.read_excel(r'C:\Users\alejandro_arboledas\OneDrive - Corporación Unificada Nacional de Educación Superior - CUN\Documentos\PYTHON\VINCULACIONES\MODELO3\2026\METAS_BOGOTA_202609.xlsx', sheet_name= periodo_especializacion)
df_metas_especializacion

,GESTOR DOCUMENTO,GESTOR CORREO,CATEGORIA,Meta,LIDER,SEDE,LOCALIDAD
0,1053814476,paula_lopezco@cun.edu.co,BRONCE,5,LUZ,BOGOTA,SOACHA
1,1012421425,diego_gonzalezar@cun.edu.co,BRONCE,5,LUZ,BOGOTA,SOACHA
2,1030576351,katherine_acosta@cun.edu.co,BRONCE,5,LUZ,BOGOTA,SOACHA
3,1014249751,jose_espitiaz@cun.edu.co,BRONCE,5,LUZ,BOGOTA,SOACHA
4,1018504409,maria_pradac@cun.edu.co,DIAMANTE,15,LUZ,BOGOTA,RAFAEL URIBE URIBE
5,1003068849,paula_vence@cun.edu.co,DIAMANTE,15,LUZ,BOGOTA,RAFAEL URIBE URIBE
6,52957207,joyce_mora@cun.edu.co,PLATA,10,LUZ,BOGOTA,RAFAEL URIBE URIBE
7,1007101612,daniel_prieto@cun.edu.co,DIAMANTE,15,LUZ,BOGOTA,RAFAEL URIBE URIBE
8,53154239,diana_sabogalpa@cun.edu.co,BRONCE,5,LUZ,BOGOTA,CIUDAD BOLIVAR
9,1013680182,sara_castanedae@cun.edu.co,PLATA,10,LUZ,BOGOTA,CIUDAD BOLIVAR


In [66]:
df_metas_especializacion = df_metas_especializacion[df_metas_especializacion['LIDER'] == lider]
df_metas_especializacion.shape

(46, 7)

In [67]:
df_metas_especializacion['GESTOR CORREO'] = df_metas_especializacion['GESTOR CORREO'].str.upper()
df_ventas_especializacion = df_pre_ventas[df_pre_ventas['Correo Ejecutivo'].isin(df_metas_especializacion['GESTOR CORREO'])].copy()
df_ventas_especializacion = df_ventas_especializacion[df_ventas_especializacion['Periodo'].isin([periodo_especializacion])]
df_ventas_especializacion.shape

(1425, 32)

In [68]:
columnas = ['Hora agregado', 'Numero de recibo','Dirección IP','Correo del estudiante','Teléfono del estudiante',
'Ciudad de residencia','Imagen del Pago Generado','% De Descuento',
'Medio de pago','Fuente de pionero','Campana','Fuente campana','Content','Sitio web', 'Tipo de inscripción',
'ID único','Fecha','Documento Ejecutivo_x','Número de aprobación de pago','Tipo de Financiación', 'Convenio', 'Valor pagado']

df_ventas_especializacion.drop(columns=columnas, inplace=True)
df_ventas_especializacion['Periodo'] = df_ventas_especializacion['Periodo'].astype(str).str.strip()
df_ventas_especializacion = df_ventas_especializacion.rename(columns={'Documento Ejecutivo_y':'Documento Ejecutivo'})
df_ventas_especializacion['Documento Ejecutivo'] = df_ventas_especializacion['Documento Ejecutivo'].astype(str)
df_ventas_especializacion['Número de Documento de estudiante'] = df_ventas_especializacion['Número de Documento de estudiante'].astype(str)
df_ventas_especializacion['Llave']=df_ventas_especializacion['Número de Documento de estudiante']+df_ventas_especializacion['Periodo']
df_ventas_especializacion.shape

(1425, 11)

SE CRUZA CON RECIBOS

In [69]:
df_recibos_especializacion = df_recibos[df_recibos['PERIODO'] == periodo_especializacion].copy()
df_recibos_especializacion.shape

(2104, 10)

In [70]:
df_recibos_especializacion = df_recibos_especializacion.merge(df_ventas_especializacion[['Propietario del registro','Modalidad','Correo Ejecutivo', 'Regional',
       'Documento Ejecutivo', 'Llave']],how='inner',left_on='LLAVE_RECIBOS',right_on='Llave')
df_recibos_especializacion.drop_duplicates(inplace=True)
df_recibos_especializacion.shape

(1416, 16)

In [71]:
#tratamiento de columnas
df_recibos_especializacion['CLASE_ACTUAL'] = df_recibos_especializacion['CLASE_ACTUAL'].astype(str).str.upper()
df_recibos_especializacion['FuerzaComercialFinal'] = df_recibos_especializacion['FuerzaComercialFinal'].str.upper()
df_recibos_especializacion['NUEVO'] = df_recibos_especializacion['NUEVO'].astype(str).str.upper()
df_recibos_especializacion['ESTADO_PAGO'] = df_recibos_especializacion['ESTADO_PAGO'].astype(str).str.upper()
df_recibos_especializacion['FuerzaComercialFinal'] = df_recibos_especializacion['FuerzaComercialFinal'].str.strip()
df_recibos_especializacion['NUEVO'] = df_recibos_especializacion['NUEVO'].str.strip()
df_recibos_especializacion['CLASE_ACTUAL'] = df_recibos_especializacion['CLASE_ACTUAL'].str.strip()
df_recibos_especializacion['ESTADO_PAGO'] = df_recibos_especializacion['ESTADO_PAGO'].str.strip()
df_recibos_especializacion['EST_MATRICULADO'] = (
    df_recibos_especializacion['EST_MATRICULADO']
    .astype(str)
    .str.strip()
    .str.upper()
)

In [72]:
# se divide entre validos
df_validos_especializacion = df_recibos_especializacion[
    ((df_recibos_especializacion['FuerzaComercialFinal'] == 'BASICA Y MEDIA') | (df_recibos_especializacion['FuerzaComercialFinal'] == 'VINCULACIONES') | (df_recibos_especializacion['FuerzaComercialFinal'] == 'INDETERMINADO')) &
    (df_recibos_especializacion['NUEVO'] == 'NUEVO') &
    (df_recibos_especializacion['EST_MATRICULADO'] == '1-ACTIVO') & 
    (df_recibos_especializacion['ESTADO_PAGO'] == 'PAGO')
].copy()
df_validos_especializacion.drop_duplicates(inplace=True)
df_validos_especializacion.shape

(1395, 16)

In [73]:
# se divide entre no validos
mask = (
    df_recibos_especializacion['FuerzaComercialFinal'].isin(
        ['BASICA Y MEDIA', 'VINCULACIONES', 'INDETERMINADO']
    ) &
    (df_recibos_especializacion['NUEVO'] == 'NUEVO') &
    (df_recibos_especializacion['EST_MATRICULADO'] == '1-ACTIVO') &
    (df_recibos_especializacion['ESTADO_PAGO'] == 'PAGO')
)
df_no_validos_especializacion = df_recibos_especializacion[~mask].copy()
df_no_validos_especializacion.drop_duplicates(inplace=True)
df_no_validos_especializacion.shape

(21, 16)

In [74]:
# Inicializar MOTIVO vacío
df_no_validos_especializacion['MOTIVO'] = ''

# Condición 1
condicion = df_no_validos_especializacion['ESTADO_PAGO'] == 'NO PAGO'
df_no_validos_especializacion.loc[condicion, 'MOTIVO'] += 'NO REGISTRA PAGO | '

# Condición 2
condicion = (
    (df_no_validos_especializacion['NUEVO'] != 'NUEVO')
)
df_no_validos_especializacion.loc[condicion, 'MOTIVO'] += 'NO ES UN ESTUDIANTE NUEVO | '

# Condición 3
condicion = df_no_validos_especializacion['FuerzaComercialFinal'] != 'VINCULACIONES'
df_no_validos_especializacion.loc[condicion, 'MOTIVO'] += 'PERTENECE A OTRA FUERZA | '

# Condición 4
condicion = df_no_validos_especializacion['EST_MATRICULADO'] != '1-ACTIVO'
df_no_validos_especializacion.loc[condicion, 'MOTIVO'] += 'MATRICULA INACTIVA | '

# Quitar el último " | "
df_no_validos_especializacion['MOTIVO'] = (
    df_no_validos_especializacion['MOTIVO']
    .str.rstrip(' |')
)

# Los que no tengan ningún motivo quedan como N/A
df_no_validos_especializacion.loc[
    df_no_validos_especializacion['MOTIVO'] == '',
    'MOTIVO'
] = 'N/A'

df_no_validos_especializacion

,DOC_ALUM,PERIODO,ESTADO_PAGO,FuerzaComercialFinal,NUEVO,CLASE_ACTUAL,TIPO,PROGRAMA,EST_MATRICULADO,LLAVE_RECIBOS,Propietario del registro,Modalidad,Correo Ejecutivo,Regional,Documento Ejecutivo,Llave,MOTIVO
171,1012378352,26ES5,PAGO,PERMANENCIA,ANTIGUO,NUEVO,Postgrado,ESPECIALIZACIÓN EN GERENCIA EDUCATIVA,1-ACTIVO,101237835226ES5,Vinculaciones Diego Alexander Gonzalez Arbelaez,Virtual,DIEGO_GONZALEZAR@CUN.EDU.CO,Bogotá,1012421425,101237835226ES5,NO ES UN ESTUDIANTE NUEVO | PERTENECE A OTRA F...
174,1012435041,26ES5,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN GERENCIA DE PROYECTOS,1-ACTIVO,101243504126ES5,Vinculaciones Natalia Remolina Herrera,Virtual,NATALIA_REMOLINAH@CUN.EDU.CO,Bogotá,1026592868,101243504126ES5,NO REGISTRA PAGO
206,1018484354,26ES5,PAGO,TELECAMPUS,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN INTELIGENCIA ARTIFICIAL (IA),1-ACTIVO,101848435426ES5,Vinculaciones Paula Alejandra Lopez Correa,Virtual,PAULA_LOPEZCO@CUN.EDU.CO,Bogotá,1053814476,101848435426ES5,PERTENECE A OTRA FUERZA
217,1022333489,26ES5,PAGO,TELECAMPUS,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN INTELIGENCIA DE NEGOCIOS,1-ACTIVO,102233348926ES5,Vinculaciones Natalia Remolina Herrera,Virtual,NATALIA_REMOLINAH@CUN.EDU.CO,Bogotá,1026592868,102233348926ES5,PERTENECE A OTRA FUERZA
280,1020724446,26ES5,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN CONTRATACIÓN ESTATAL,1-ACTIVO,102072444626ES5,Vinculaciones Sara Valentina Castañeda Espitia,Virtual,SARA_CASTANEDAE@CUN.EDU.CO,Bogotá,1014187111,102072444626ES5,NO REGISTRA PAGO
324,1023032719,26ES5,PAGO,PERMANENCIA,ANTIGUO,NUEVO,Postgrado,ESPECIALIZACIÓN EN DESARROLLO ORGANIZACIONAL Y...,1-ACTIVO,102303271926ES5,Vinculaciones Diana Clemencia Sabogal Parrado,Virtual,DIANA_SABOGALPA@CUN.EDU.CO,Bogotá,53154239,102303271926ES5,NO ES UN ESTUDIANTE NUEVO | PERTENECE A OTRA F...
434,1030702210,26ES5,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN CONTRATACIÓN ESTATAL,1-ACTIVO,103070221026ES5,Vinculaciones Carolina Rocio Viloria Pedreros,Virtual,CAROLINA_VILORIAP@CUN.EDU.CO,Bogotá,1015393814,103070221026ES5,NO REGISTRA PAGO
438,1031164593,26ES5,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN SEGURIDAD Y SALUD EN EL TRA...,1-ACTIVO,103116459326ES5,Vinculaciones Joyce Lorena Mora Solorzano,Virtual,JOYCE_MORA@CUN.EDU.CO,Bogotá,52957207,103116459326ES5,NO REGISTRA PAGO
439,1054679821,26ES5,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN GERENCIA FINANCIERA,1-ACTIVO,105467982126ES5,Vinculaciones Dairon Alejandro Ibañez Rey,Virtual,DAIRON_IBANEZR@CUN.EDU.CO,Bogotá,1019115249,105467982126ES5,NO REGISTRA PAGO
442,1061735942,26ES5,PAGO,CONTACT,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN DESARROLLO ORGANIZACIONAL Y...,1-ACTIVO,106173594226ES5,Vinculaciones Natalia Remolina Herrera,Virtual,NATALIA_REMOLINAH@CUN.EDU.CO,Bogotá,1026592868,106173594226ES5,PERTENECE A OTRA FUERZA


In [75]:
# se traen las ventas que no tienen marca de pago
df_no_recibos_especializacion = df_ventas_especializacion[~df_ventas_especializacion['Llave'].isin(df_recibos_especializacion['LLAVE_RECIBOS'])].copy()
df_no_recibos_especializacion['DOC_ALUM'] = df_no_recibos_especializacion['Número de Documento de estudiante'].astype(str)
df_no_recibos_especializacion['PERIODO'] = df_no_recibos_especializacion['Periodo']
df_no_recibos_especializacion['ESTADO_PAGO'] = ' '
df_no_recibos_especializacion['EST_MATRICULADO'] = ' '
df_no_recibos_especializacion['FuerzaComercialFinal'] = ' '
df_no_recibos_especializacion['NUEVO'] = ' '
df_no_recibos_especializacion['CLASE_ACTUAL'] = ' '
df_no_recibos_especializacion['TIPO'] = ' '
df_no_recibos_especializacion['PROGRAMA'] = df_no_recibos_especializacion['Programa vendido'] 
df_no_recibos_especializacion['LLAVE_RECIBOS'] = ' '
df_no_recibos_especializacion['MOTIVO'] = 'NO CUENTA CON MARCA DE PAGO'
df_no_recibos_especializacion = df_no_recibos_especializacion[['DOC_ALUM', 'PERIODO', 'ESTADO_PAGO', 'FuerzaComercialFinal', 'NUEVO',
       'CLASE_ACTUAL', 'TIPO', 'PROGRAMA', 'EST_MATRICULADO', 'LLAVE_RECIBOS',
       'Propietario del registro', 'Modalidad', 'Correo Ejecutivo', 'Regional',
       'Documento Ejecutivo', 'Llave','MOTIVO']]
df_no_recibos_especializacion.shape

(15, 17)

In [76]:
# SE UNIFICAN LOS NO VALIDOS Y LOS QUE NO TIENEN MARCA DE PAGO
df_no_validos_especializacion_2 = pd.concat([df_no_validos_especializacion,df_no_recibos_especializacion])
df_no_validos_especializacion_2

,DOC_ALUM,PERIODO,ESTADO_PAGO,FuerzaComercialFinal,NUEVO,CLASE_ACTUAL,TIPO,PROGRAMA,EST_MATRICULADO,LLAVE_RECIBOS,Propietario del registro,Modalidad,Correo Ejecutivo,Regional,Documento Ejecutivo,Llave,MOTIVO
171,1012378352,26ES5,PAGO,PERMANENCIA,ANTIGUO,NUEVO,Postgrado,ESPECIALIZACIÓN EN GERENCIA EDUCATIVA,1-ACTIVO,101237835226ES5,Vinculaciones Diego Alexander Gonzalez Arbelaez,Virtual,DIEGO_GONZALEZAR@CUN.EDU.CO,Bogotá,1012421425,101237835226ES5,NO ES UN ESTUDIANTE NUEVO | PERTENECE A OTRA F...
174,1012435041,26ES5,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN GERENCIA DE PROYECTOS,1-ACTIVO,101243504126ES5,Vinculaciones Natalia Remolina Herrera,Virtual,NATALIA_REMOLINAH@CUN.EDU.CO,Bogotá,1026592868,101243504126ES5,NO REGISTRA PAGO
206,1018484354,26ES5,PAGO,TELECAMPUS,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN INTELIGENCIA ARTIFICIAL (IA),1-ACTIVO,101848435426ES5,Vinculaciones Paula Alejandra Lopez Correa,Virtual,PAULA_LOPEZCO@CUN.EDU.CO,Bogotá,1053814476,101848435426ES5,PERTENECE A OTRA FUERZA
217,1022333489,26ES5,PAGO,TELECAMPUS,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN INTELIGENCIA DE NEGOCIOS,1-ACTIVO,102233348926ES5,Vinculaciones Natalia Remolina Herrera,Virtual,NATALIA_REMOLINAH@CUN.EDU.CO,Bogotá,1026592868,102233348926ES5,PERTENECE A OTRA FUERZA
280,1020724446,26ES5,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN CONTRATACIÓN ESTATAL,1-ACTIVO,102072444626ES5,Vinculaciones Sara Valentina Castañeda Espitia,Virtual,SARA_CASTANEDAE@CUN.EDU.CO,Bogotá,1014187111,102072444626ES5,NO REGISTRA PAGO
324,1023032719,26ES5,PAGO,PERMANENCIA,ANTIGUO,NUEVO,Postgrado,ESPECIALIZACIÓN EN DESARROLLO ORGANIZACIONAL Y...,1-ACTIVO,102303271926ES5,Vinculaciones Diana Clemencia Sabogal Parrado,Virtual,DIANA_SABOGALPA@CUN.EDU.CO,Bogotá,53154239,102303271926ES5,NO ES UN ESTUDIANTE NUEVO | PERTENECE A OTRA F...
434,1030702210,26ES5,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN CONTRATACIÓN ESTATAL,1-ACTIVO,103070221026ES5,Vinculaciones Carolina Rocio Viloria Pedreros,Virtual,CAROLINA_VILORIAP@CUN.EDU.CO,Bogotá,1015393814,103070221026ES5,NO REGISTRA PAGO
438,1031164593,26ES5,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN SEGURIDAD Y SALUD EN EL TRA...,1-ACTIVO,103116459326ES5,Vinculaciones Joyce Lorena Mora Solorzano,Virtual,JOYCE_MORA@CUN.EDU.CO,Bogotá,52957207,103116459326ES5,NO REGISTRA PAGO
439,1054679821,26ES5,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN GERENCIA FINANCIERA,1-ACTIVO,105467982126ES5,Vinculaciones Dairon Alejandro Ibañez Rey,Virtual,DAIRON_IBANEZR@CUN.EDU.CO,Bogotá,1019115249,105467982126ES5,NO REGISTRA PAGO
442,1061735942,26ES5,PAGO,CONTACT,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN DESARROLLO ORGANIZACIONAL Y...,1-ACTIVO,106173594226ES5,Vinculaciones Natalia Remolina Herrera,Virtual,NATALIA_REMOLINAH@CUN.EDU.CO,Bogotá,1026592868,106173594226ES5,PERTENECE A OTRA FUERZA


SE CRUZA CON OPORTUNIDADES

In [77]:
# se crea la llave para cruzar por doc alum+periodo+correo ejecutivo
df_validos_especializacion['LLAVE_ZOHO'] = df_validos_especializacion['DOC_ALUM']+df_validos_especializacion['PERIODO']+df_validos_especializacion['Correo Ejecutivo']

In [78]:
df_zoho_recibos_especializacion = df_validos_especializacion.merge(df_zoho_vinculaciones[['Propietario_de_Oportunidad_Name','Fase','Hora_de_modificación','LLAVE_ZOHO']],left_on='LLAVE_ZOHO',right_on='LLAVE_ZOHO',how='inner').copy()
df_zoho_recibos_especializacion.drop_duplicates(inplace=True)
df_zoho_recibos_especializacion.shape

(1259, 20)

In [79]:
df_no_validos_zoho_especializacion = df_validos_especializacion[~df_validos_especializacion['LLAVE_ZOHO'].isin(df_zoho_vinculaciones['LLAVE_ZOHO'])].copy()
df_no_validos_zoho_especializacion['MOTIVO'] = 'NO ESTA OK EN ZOHO'
df_no_validos_zoho_especializacion = df_no_validos_zoho_especializacion.drop(columns=['LLAVE_ZOHO'])
df_no_validos_zoho_especializacion.shape

(144, 17)

In [80]:
# SE UNIFICAN LOS NO VALIDOS TOTALES DE VIRTUALES
df_no_validos_especializacion_3 = pd.concat([df_no_validos_especializacion_2,df_no_validos_zoho_especializacion])
df_no_validos_especializacion_3

,DOC_ALUM,PERIODO,ESTADO_PAGO,FuerzaComercialFinal,NUEVO,CLASE_ACTUAL,TIPO,PROGRAMA,EST_MATRICULADO,LLAVE_RECIBOS,Propietario del registro,Modalidad,Correo Ejecutivo,Regional,Documento Ejecutivo,Llave,MOTIVO
171,1012378352,26ES5,PAGO,PERMANENCIA,ANTIGUO,NUEVO,Postgrado,ESPECIALIZACIÓN EN GERENCIA EDUCATIVA,1-ACTIVO,101237835226ES5,Vinculaciones Diego Alexander Gonzalez Arbelaez,Virtual,DIEGO_GONZALEZAR@CUN.EDU.CO,Bogotá,1012421425,101237835226ES5,NO ES UN ESTUDIANTE NUEVO | PERTENECE A OTRA F...
174,1012435041,26ES5,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN GERENCIA DE PROYECTOS,1-ACTIVO,101243504126ES5,Vinculaciones Natalia Remolina Herrera,Virtual,NATALIA_REMOLINAH@CUN.EDU.CO,Bogotá,1026592868,101243504126ES5,NO REGISTRA PAGO
206,1018484354,26ES5,PAGO,TELECAMPUS,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN INTELIGENCIA ARTIFICIAL (IA),1-ACTIVO,101848435426ES5,Vinculaciones Paula Alejandra Lopez Correa,Virtual,PAULA_LOPEZCO@CUN.EDU.CO,Bogotá,1053814476,101848435426ES5,PERTENECE A OTRA FUERZA
217,1022333489,26ES5,PAGO,TELECAMPUS,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN INTELIGENCIA DE NEGOCIOS,1-ACTIVO,102233348926ES5,Vinculaciones Natalia Remolina Herrera,Virtual,NATALIA_REMOLINAH@CUN.EDU.CO,Bogotá,1026592868,102233348926ES5,PERTENECE A OTRA FUERZA
280,1020724446,26ES5,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN CONTRATACIÓN ESTATAL,1-ACTIVO,102072444626ES5,Vinculaciones Sara Valentina Castañeda Espitia,Virtual,SARA_CASTANEDAE@CUN.EDU.CO,Bogotá,1014187111,102072444626ES5,NO REGISTRA PAGO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1396,79171257,26ES5,PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN GERENCIA DE PROYECTOS,1-ACTIVO,7917125726ES5,Vinculaciones Erika Andrea Ramirez Rico,Virtual,ERIKA_RAMIREZ@CUN.EDU.CO,Bogotá,1012382480,7917125726ES5,NO ESTA OK EN ZOHO
1405,79974945,26ES5,PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN CONTRATACIÓN ESTATAL,1-ACTIVO,7997494526ES5,Vinculaciones Yudi Esperanza Barrero Romero,Virtual,YUDI_BARREROR@CUN.EDU.CO,Bogotá,1023868232,7997494526ES5,NO ESTA OK EN ZOHO
1406,79989129,26ES5,PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN CIBERSEGURIDAD,1-ACTIVO,7998912926ES5,Vinculaciones Erika Andrea Ramirez Rico,Virtual,ERIKA_RAMIREZ@CUN.EDU.CO,Bogotá,1012382480,7998912926ES5,NO ESTA OK EN ZOHO
1415,80812956,26ES5,PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACION EN GESTION DE TECNOLOGIAS DE L...,1-ACTIVO,8081295626ES5,Vinculaciones Erika Andrea Ramirez Rico,Virtual,ERIKA_RAMIREZ@CUN.EDU.CO,Bogotá,1012382480,8081295626ES5,NO ESTA OK EN ZOHO


SE CRUZA CON EL HISTORICO

In [81]:
df_validos_especializaciones_final = df_zoho_recibos_especializacion[~df_zoho_recibos_especializacion['Llave'].isin(df_historico['LLAVE_HISTORICO'])].copy()
df_validos_especializaciones_final.shape

(1259, 20)

In [82]:
df_no_validos_historico_especializaciones = df_zoho_recibos_presencial[df_zoho_recibos_presencial['Llave'].isin(df_historico['LLAVE_HISTORICO'])].copy()
df_no_validos_historico_especializaciones['MOTIVO'] = 'YA SE PAGO EN LA LIQUIDACION PASADA' 
df_no_validos_historico_especializaciones = df_no_validos_historico_especializaciones.drop(columns=['LLAVE_ZOHO','Propietario_de_Oportunidad_Name','Fase','Hora_de_modificación'])
df_no_validos_historico_especializaciones.shape

(0, 17)

SE UNIFICAN LOS NO VALIDOS ESPECIALIZACION

In [83]:
df_novalidos_especializacion_final = pd.concat([df_no_validos_especializacion_3,df_no_validos_historico_especializaciones])
df_novalidos_especializacion_final.shape

(180, 17)

SE INICIAN CALCULOS DE COMISION

In [84]:
# Filtrar y contar las ventas por estado de pago (PAGO y NO PAGO)
ventas_asesor_especializaciones = (
    df_validos_especializaciones_final
    .groupby(["Documento Ejecutivo"])
    .size()
    .reset_index(name="VENTAS_ESPECIALIZACION")
).copy()

In [85]:
df_metas_especializacion['GESTOR DOCUMENTO'] = df_metas_especializacion['GESTOR DOCUMENTO'].astype(str)
ventas_asesor_especializaciones['Documento Ejecutivo'] = ventas_asesor_especializaciones['Documento Ejecutivo'].astype(str)
ventas_asesor_especializaciones = ventas_asesor_especializaciones.merge(df_metas_especializacion,how='left',left_on='Documento Ejecutivo',right_on='GESTOR DOCUMENTO').copy()
ventas_asesor_especializaciones = ventas_asesor_especializaciones.drop(columns=['GESTOR DOCUMENTO'])
ventas_asesor_especializaciones

,Documento Ejecutivo,VENTAS_ESPECIALIZACION,GESTOR CORREO,CATEGORIA,Meta,LIDER,SEDE,LOCALIDAD
0,1000335802,6,ANGIE_DIAZJ@CUN.EDU.CO,ESMERALDA,20.0,LUZ,BOGOTA,GRANDES CUENTAS
1,1000457601,9,DIEGO_BOHORQUEZCO@CUN.EDU.CO,PLATA,10.0,LUZ,BOGOTA,ENGATIVA
2,1001056960,12,NaN,NaN,NaN,NaN,NaN,NaN
3,1001956960,12,PAULA_HURTADOM@CUN.EDU.CO,BRONCE,5.0,LUZ,BOGOTA,KENNEDY
4,1003068849,15,PAULA_VENCE@CUN.EDU.CO,DIAMANTE,15.0,LUZ,BOGOTA,RAFAEL URIBE URIBE
5,1003515891,2,BRESLY_MEDINARI@CUN.EDU.CO,JUNIOR,20.0,LUZ,BOGOTA,GRANDES CUENTAS
6,1006510805,1,KARLA_REYES@CUN.EDU.CO,BRONCE,5.0,LUZ,BOGOTA,ENGATIVA
7,1007101612,85,DANIEL_PRIETO@CUN.EDU.CO,DIAMANTE,15.0,LUZ,BOGOTA,RAFAEL URIBE URIBE
8,1007702093,14,ANGGY_AGRAYVI@CUN.EDU.CO,HIERRO,10.0,LUZ,BOGOTA,SABANA OCCIDENTE
9,1012388460,65,DAIRON_TELLEZZA@CUN.EDU.CO,PLATA,10.0,LUZ,BOGOTA,BOSA


In [86]:
esquema_comisiones_especializaciones = {

    "LUZ": {
        "ORO": {
                        25: [  
                                (25, float('inf'), 80000)
                            ]},

        "ESMERALDA": {
                        20: [  
                                (20, 24, 70000),
                                (25, float('inf'), 80000)
                            ]},
        "DIAMANTE": {
                        15: [  
                                (15, 19, 60000),
                                (20, 24, 70000),
                                (25, float('inf'), 80000)
                            ]},
        "PLATA": {
            
                        10: [  
                                (10, 14, 50000),    
                                (15, 19, 60000),
                                (20, 24, 70000),
                                (25, float('inf'), 80000)
                            ]},
        "BRONCE": {
            
                        5: [  
                                (5, 9, 40000),
                                (10, 14, 50000),    
                                (15, 19, 60000),
                                (20, 24, 70000),
                                (25, float('inf'), 80000)
                            ]},
        "HIERRO": {
            
                        10: [  
                                (10, 14, 40000),    
                                (15, 19, 60000),
                                (20, 24, 70000),
                                (25, float('inf'), 80000)
                            ]}                               
    }}

In [87]:
def asignar_comision_especializaciones(lider, categoria, meta, ventas):

    # 1️⃣ Validar líder
    if lider not in esquema_comisiones_especializaciones:
        return 0

    # 2️⃣ Validar sede
    if categoria not in esquema_comisiones_especializaciones[lider]:
        return 0
    # 4️⃣ Validar meta
    if meta not in esquema_comisiones_especializaciones[lider][categoria]:
        return 0

    # 5️⃣ Validar cumplimiento de meta
    if ventas < meta:
        return 0

    # 6️⃣ Obtener los pisos correspondientes a la meta
    rangos = esquema_comisiones_especializaciones[lider][categoria][meta]

    # 7️⃣ Buscar el piso correspondiente según las ventas
    for min_rango, max_rango, comision in rangos:

        if min_rango <= ventas <= max_rango:

            # Comisión por venta × número de ventas
            return ventas * comision

    return 0

In [88]:
ventas_asesor_especializaciones["COMISION_TOTAL_ESPECIALIZACION"] = ventas_asesor_especializaciones.apply(
    lambda row: asignar_comision_especializaciones(
        row["LIDER"],
        row["CATEGORIA"],
        row["Meta"],
        row["VENTAS_ESPECIALIZACION"]
    ),
    axis=1
)
ventas_asesor_especializaciones['COMISION_UNITARIA_ESPECIALIZACION'] = ventas_asesor_especializaciones['COMISION_TOTAL_ESPECIALIZACION'] / ventas_asesor_especializaciones['VENTAS_ESPECIALIZACION']
ventas_asesor_especializaciones['%CUMPLIMIENTO_ESPECIALIZACION'] = (
    (ventas_asesor_especializaciones['VENTAS_ESPECIALIZACION'] / ventas_asesor_especializaciones['Meta']) * 100).round(2).astype(str) + '%'
ventas_asesor_especializaciones

,Documento Ejecutivo,VENTAS_ESPECIALIZACION,GESTOR CORREO,CATEGORIA,Meta,LIDER,SEDE,LOCALIDAD,COMISION_TOTAL_ESPECIALIZACION,COMISION_UNITARIA_ESPECIALIZACION,%CUMPLIMIENTO_ESPECIALIZACION
0,1000335802,6,ANGIE_DIAZJ@CUN.EDU.CO,ESMERALDA,20.0,LUZ,BOGOTA,GRANDES CUENTAS,0,0.0,30.0%
1,1000457601,9,DIEGO_BOHORQUEZCO@CUN.EDU.CO,PLATA,10.0,LUZ,BOGOTA,ENGATIVA,0,0.0,90.0%
2,1001056960,12,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,nan%
3,1001956960,12,PAULA_HURTADOM@CUN.EDU.CO,BRONCE,5.0,LUZ,BOGOTA,KENNEDY,600000,50000.0,240.0%
4,1003068849,15,PAULA_VENCE@CUN.EDU.CO,DIAMANTE,15.0,LUZ,BOGOTA,RAFAEL URIBE URIBE,900000,60000.0,100.0%
5,1003515891,2,BRESLY_MEDINARI@CUN.EDU.CO,JUNIOR,20.0,LUZ,BOGOTA,GRANDES CUENTAS,0,0.0,10.0%
6,1006510805,1,KARLA_REYES@CUN.EDU.CO,BRONCE,5.0,LUZ,BOGOTA,ENGATIVA,0,0.0,20.0%
7,1007101612,85,DANIEL_PRIETO@CUN.EDU.CO,DIAMANTE,15.0,LUZ,BOGOTA,RAFAEL URIBE URIBE,6800000,80000.0,566.67%
8,1007702093,14,ANGGY_AGRAYVI@CUN.EDU.CO,HIERRO,10.0,LUZ,BOGOTA,SABANA OCCIDENTE,560000,40000.0,140.0%
9,1012388460,65,DAIRON_TELLEZZA@CUN.EDU.CO,PLATA,10.0,LUZ,BOGOTA,BOSA,5200000,80000.0,650.0%


SE ARMA EL DATAFRAME FINAL

In [89]:
ventas_asesor_virtuales = ventas_asesor_virtuales.rename(columns={'Meta':'META_VIRTUAL'})
ventas_asesor_presenciales = ventas_asesor_presenciales.rename(columns={'Meta':'META_PRESENCIAL'})
ventas_asesor_especializaciones = ventas_asesor_especializaciones.rename(columns={'Meta':'META_ESPECIALIZACION'})
base_pago = ventas_asesor_virtuales.merge(
    ventas_asesor_presenciales[
        [
            'Documento Ejecutivo',
            'VENTAS_PRESENCIALES',
            'META_PRESENCIAL',
            'COMISION_TOTAL_PRESENCIAL',
            'COMISION_UNITARIA_PRESENCIAL',
            '%CUMPLIMIENTO_PRESENCIAL'
        ]
    ],
    how='left',
    on='Documento Ejecutivo'
).copy()


In [90]:
base_pago_final = base_pago.merge(
    ventas_asesor_especializaciones[
        [
       'Documento Ejecutivo', 'VENTAS_ESPECIALIZACION','META_ESPECIALIZACION', 
       'COMISION_TOTAL_ESPECIALIZACION', 'COMISION_UNITARIA_ESPECIALIZACION',
       '%CUMPLIMIENTO_ESPECIALIZACION'
        ]
    ],
    how='left',
    on='Documento Ejecutivo'
).copy()

In [91]:
columnas_ventas = [
    'VENTAS_VIRTUALES',
    'VENTAS_PRESENCIALES',
    'VENTAS_ESPECIALIZACION'
]

columnas_comisiones = [
    'COMISION_TOTAL_VIRTUAL',
    'COMISION_TOTAL_PRESENCIAL',
    'COMISION_TOTAL_ESPECIALIZACION'
]

base_pago_final['VENTAS_TOTALES'] = (
    base_pago_final[columnas_ventas].fillna(0).sum(axis=1)
)

base_pago_final['COMISION_FINAL'] = (
    base_pago_final[columnas_comisiones].fillna(0).sum(axis=1)
)
base_pago_final['PERIODO_PAGO'] = periodo_pago
base_pago_final = base_pago_final.rename(columns=['CATEGORIA':'CARGO'])
base_pago_final = base_pago_final[['Documento Ejecutivo','GESTOR CORREO', 'CARGO', 'LIDER', 'SEDE', 'LOCALIDAD', 'PERIODO_PAGO','VENTAS_VIRTUALES', 
       'META_VIRTUAL','%CUMPLIMIENTO_VIRTUAL', 'COMISION_UNITARIA_VIRTUAL','COMISION_TOTAL_VIRTUAL',
       'VENTAS_PRESENCIALES', 'META_PRESENCIAL','%CUMPLIMIENTO_PRESENCIAL','COMISION_UNITARIA_PRESENCIAL','COMISION_TOTAL_PRESENCIAL',
       'VENTAS_ESPECIALIZACION', 'META_ESPECIALIZACION', '%CUMPLIMIENTO_ESPECIALIZACION', 'COMISION_UNITARIA_ESPECIALIZACION',
       'COMISION_TOTAL_ESPECIALIZACION','VENTAS_TOTALES', 'COMISION_FINAL']]

SyntaxError: invalid syntax (2765992337.py, line 21)

In [ ]:
base_pago_final

,Documento Ejecutivo,GESTOR CORREO,CATEGORIA,LIDER,SEDE,LOCALIDAD,PERIODO_PAGO,VENTAS_VIRTUALES,META_VIRTUAL,%CUMPLIMIENTO_VIRTUAL,...,%CUMPLIMIENTO_PRESENCIAL,COMISION_UNITARIA_PRESENCIAL,COMISION_TOTAL_PRESENCIAL,VENTAS_ESPECIALIZACION,META_ESPECIALIZACION,%CUMPLIMIENTO_ESPECIALIZACION,COMISION_UNITARIA_ESPECIALIZACION,COMISION_TOTAL_ESPECIALIZACION,VENTAS_TOTALES,COMISION_FINAL
0,1000335802,ANGIE_DIAZJ@CUN.EDU.CO,ESMERALDA,LUZ,BOGOTA,GRANDES CUENTAS,202609,6,50.0,12.0%,...,15.0%,0.0,0.0,6.0,20.0,30.0%,0.0,0.0,15.0,0.0
1,1000457601,DIEGO_BOHORQUEZCO@CUN.EDU.CO,PLATA,LUZ,BOGOTA,ENGATIVA,202609,14,30.0,46.67%,...,NaN,NaN,NaN,9.0,10.0,90.0%,0.0,0.0,23.0,0.0
2,1001056960,NaN,NaN,NaN,NaN,NaN,202609,5,NaN,nan%,...,nan%,0.0,0.0,12.0,NaN,nan%,0.0,0.0,19.0,0.0
3,1001956960,PAULA_HURTADOM@CUN.EDU.CO,BRONCE,LUZ,BOGOTA,KENNEDY,202609,5,20.0,25.0%,...,40.0%,0.0,0.0,12.0,5.0,240.0%,50000.0,600000.0,19.0,600000.0
4,1003068849,PAULA_VENCE@CUN.EDU.CO,DIAMANTE,LUZ,BOGOTA,RAFAEL URIBE URIBE,202609,30,40.0,75.0%,...,20.0%,0.0,0.0,15.0,15.0,100.0%,60000.0,900000.0,48.0,900000.0
5,1007101612,DANIEL_PRIETO@CUN.EDU.CO,DIAMANTE,LUZ,BOGOTA,RAFAEL URIBE URIBE,202609,44,40.0,110.0%,...,60.0%,0.0,0.0,85.0,15.0,566.67%,80000.0,6800000.0,138.0,9440000.0
6,1007702093,ANGGY_AGRAYVI@CUN.EDU.CO,HIERRO,LUZ,BOGOTA,SABANA OCCIDENTE,202609,10,10.0,100.0%,...,NaN,NaN,NaN,14.0,10.0,140.0%,40000.0,560000.0,24.0,960000.0
7,1007812062,JUAN_CASTANEDAA@CUN.EDU.CO,ESMERALDA,LUZ,BOGOTA,GRANDES CUENTAS,202609,1,50.0,2.0%,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0
8,1012388460,DAIRON_TELLEZZA@CUN.EDU.CO,PLATA,LUZ,BOGOTA,BOSA,202609,55,30.0,183.33%,...,10.0%,0.0,0.0,65.0,10.0,650.0%,80000.0,5200000.0,121.0,9050000.0
9,1012421425,DIEGO_GONZALEZAR@CUN.EDU.CO,BRONCE,LUZ,BOGOTA,SOACHA,202609,24,20.0,120.0%,...,NaN,NaN,NaN,78.0,5.0,1560.0%,80000.0,6240000.0,102.0,7200000.0


In [ ]:
no_validos_total = pd.concat([df_novalidos_virtual_final,df_novalidos_presencial_final,df_novalidos_especializacion_final])
no_validos_total['PERIODO_PAGO'] = periodo_pago
no_validos_total = no_validos_total[['DOC_ALUM', 'PERIODO', 'ESTADO_PAGO', 'FuerzaComercialFinal', 'NUEVO',
       'CLASE_ACTUAL', 'TIPO', 'PROGRAMA', 'EST_MATRICULADO',
       'Propietario del registro', 'Modalidad', 'Correo Ejecutivo','Documento Ejecutivo', 'Regional',
       'PERIODO_PAGO', 'MOTIVO']]
no_validos_total.shape

(404, 16)

In [ ]:
validos_total = pd.concat([df_validos_virtuales_final,df_validos_presenciales_final,df_validos_especializaciones_final])
validos_total['PERIODO_PAGO'] = periodo_pago
validos_total = validos_total[['DOC_ALUM', 'PERIODO', 'ESTADO_PAGO', 'FuerzaComercialFinal', 'NUEVO',
       'CLASE_ACTUAL', 'TIPO', 'PROGRAMA', 'EST_MATRICULADO',
       'Propietario del registro', 'Modalidad', 'Correo Ejecutivo', 'Regional',
       'Documento Ejecutivo','Propietario_de_Oportunidad_Name', 'Fase', 'Hora_de_modificación', 'PERIODO_PAGO']]

In [98]:
with pd.ExcelWriter(
    f"LIQUIDACION_VINCULACIONES_BOGOTA_{periodo_pago}.xlsx",
    engine="openpyxl"
) as writer:

    base_pago_final.to_excel(
        writer,
        sheet_name="Comisiones_Asesores",
        index=False
    )

    validos_total.to_excel(
        writer,
        sheet_name="Validos",
        index=False
    )

    no_validos_total.to_excel(
        writer,
        sheet_name="No_Validas",
        index=False
    )

In [97]:
RR

NameError: name 'RR' is not defined

mal

In [ ]:
RR

DBAPIError: (pyodbc.Error) ('01000', '[01000] [Microsoft][ODBC SQL Server Driver][DBNETLIB]ConnectionWrite (send()). (10054) (SQLExecDirectW); [01000] [Microsoft][ODBC SQL Server Driver][DBNETLIB]Error general de red. Consulte la documentación de la red. (11)')
[SQL:  

        
        SELECT identificacion_btpr,concat(nombre_btpr,' ',segundo_nombre_btpr,' ',primer_apellido_btpr,' ',
                        segundo_apellido_btpr) as Nombre
        FROM HUB_persona
 ]
(Background on this error at: https://sqlalche.me/e/20/dbapi)

In [ ]:
df_persona=df_persona.drop_duplicates()
df_persona.shape

(665174, 2)

In [ ]:
df_final=pd.merge(df_piso,df_persona[['Nombre','identificacion_btpr']],how='left',left_on='Documento Ejecutivo',
                  right_on='identificacion_btpr')
df_final=df_final.drop(columns='identificacion_btpr')
df_final

,Documento_Estudiante,PERIODO_y,Nombre_de_Oportunidad,Propietario_de_Oportunidad_Name,Hora_de_creación,Fecha_de_cierre,Propietario_de_Oportunidad,Fase,Hora_de_modificación,Programa_de_interés,...,EST_MATRICULADO,_merge,VENTAS_ASESOR,Refe,Ventas Totales,Meta,LIDER,SEDE,CARGO,Nombre


In [ ]:
##df_final.loc[(df_final['Documento Ejecutivo'] == '1143244777'),'Meta']=1
##df_final.loc[(df_final['Documento Ejecutivo'] == '1081796706'),'Meta']=1
##df_final.loc[(df_final['Documento Ejecutivo'] == '44156610'),'Meta']=1
##df_final.loc[(df_final['Documento Ejecutivo'] == '1110553023'),'Meta']=1
##df_final.loc[(df_final['Documento Ejecutivo'] == '1112496766'),'Meta']=1


In [ ]:
### Definir los rangos de comisión
##rangos = [
##    (1,15,23000),
##    (15,20,80000), 
##    (20,30,90000),
##    (31, float('inf'), 110000)
##]
###rangos = [
###    (1,15,23000), 
###    (15,20,80000), #ELIMINAR DESPUÉS
###    (21, float('inf'), 120000)
###]
##
### Función para asignar la comisión según el número de pagos
##def asignar_comision(piso,pagos):
##    if piso == 0:
##        return 0
##    else:
##        if pagos < piso:
##            return 0
##        else:
##            for min_rango, max_rango, comision in rangos:
##                if min_rango <= pagos <= max_rango:
##                    return comision
##    return 0  # Si no entra en ningún rango (caso raro)
##
### Aplicar la función al DataFrame
##df_final["Comision Unitaria"] = df_final.apply(lambda row: asignar_comision(row['Meta'], row['Ventas Totales']), axis=1)


In [ ]:
esquema_comisiones = {

    "SANDRA": {
        "26V03": {
            "Gestor de ventas Senior":{ 
                        33: [  
                                (33, 39, 70000),
                                (40, float('inf'), 100000)
                            ]},
            "Gestor de ventas Junior":{ 
                        18: [  
                                (18, 24, 70000),
                                (25, float('inf'), 100000)
                            
                            ]       
            },

            "Gestor Grandes Cuentas":{ 
                        60: [  
                                (60, 79, 80000),
                                (80, float('inf'), 100000)
                            ]}},

        "26ES3": {

           "Gestor de ventas Senior":{ 
                        20: [  
                                (20, 29, 70000),
                                (30, float('inf'), 100000)
                            ]},
            "Gestor de ventas Junior":{ 
                        10: [  
                                (10, 14, 70000),
                                (15, float('inf'), 100000)
                            ]       
            },

            "Gestor Grandes Cuentas":{ 
                        20: [  
                                (20, 29, 65000),
                                (30, float('inf'), 80000)
                            ]}},
        "26T03": {

            "Gestor de ventas Senior":{ 
                        1: [  
                               (1, float('inf'), 70000)
                            ]},
            "Gestor de ventas Junior":{ 
                        1: [  
                                (1, float('inf'), 70000)
                            
                            ]       
            },

            "Gestor Grandes Cuentas":{ 
                        1: [   
                                (1, float('inf'), 70000)
                            ]}}
    }}

In [ ]:
def asignar_comision(lider, periodo, cargo, meta,ventas):

    # 1️⃣ Validar líder
    if lider not in esquema_comisiones:
        return 0

    # 2️⃣ Validar periodo
    if periodo not in esquema_comisiones[lider]:
        return 0

    # 3️⃣ Validar sede
    if cargo not in esquema_comisiones[lider][periodo]:
        return 0
    # 5️⃣ Validar cumplimiento de meta
    if ventas < meta:
        return 0

    rangos = esquema_comisiones[lider][periodo][cargo][meta]

    # 4️⃣ Evaluar rangos según ventas
    for min_rango, max_rango, comision in rangos:
        if min_rango <= ventas <= max_rango:
            return comision

    return 0


In [ ]:
df_final["Comision Unitaria"] = df_final.apply(
    lambda row: asignar_comision(
        row["LIDER "],
        row["PERIODO_y"],
        row["CARGO"],
        row["Meta"],
        row["Ventas Totales"]
    ),
    axis=1
)



In [ ]:
df_final.loc[df_final['ESTADO_PAGO']=='NO PAGO','Comision Unitaria'] = 0

In [ ]:
#Calcular la comision total
df_final['Comision Total'] = df_final['Ventas Totales']*df_final['Comision Unitaria']

In [ ]:
df_final.columns

Index(['Documento_Estudiante', 'PERIODO_y', 'Nombre_de_Oportunidad',
       'Propietario_de_Oportunidad_Name', 'Hora_de_creación',
       'Fecha_de_cierre', 'Propietario_de_Oportunidad', 'Fase',
       'Hora_de_modificación', 'Programa_de_interés', 'PERIODO_x', 'Id',
       'Nombre_de_Interesado', 'Creado_por', 'Hora_de_la_última_actividad',
       'Correo_electrónico', 'llave1', 'Documento Ejecutivo', 'ESTADO_PAGO',
       'FuerzaComercialFinal', 'FEC_PAGO_LIQ', 'NUEVO', 'CLASE_ACTUAL', 'TIPO',
       'PROGRAMA', 'EST_MATRICULADO', '_merge', 'VENTAS_ASESOR', 'Refe',
       'Ventas Totales', 'Meta', 'LIDER ', 'SEDE', 'CARGO', 'Nombre',
       'Comision Unitaria', 'Comision Total'],
      dtype='object')

In [ ]:
df_asesores = df_final[['Nombre','Documento Ejecutivo','Ventas Totales', 'Meta',
            'Comision Unitaria', 'Comision Total' ]]
df_asesores.drop_duplicates(subset='Nombre')
df_asesores

,Nombre,Documento Ejecutivo,Ventas Totales,Meta,Comision Unitaria,Comision Total


In [ ]:
df_asesores_2 = df_final.groupby('Documento Ejecutivo', as_index=False).agg({
    'Nombre': 'first',               # Primer nombre encontrado por documento
    'Ventas Totales': 'first',         # Suma total de ventas
    'Meta': 'mean',                   # Suma total de metas
    'Comision Unitaria' : 'first',
    'Comision Total' : 'first'
})


In [ ]:
df_asesores_2

,Documento Ejecutivo,Nombre,Ventas Totales,Meta,Comision Unitaria,Comision Total


In [ ]:
df_final

,Documento_Estudiante,PERIODO_y,Nombre_de_Oportunidad,Propietario_de_Oportunidad_Name,Hora_de_creación,Fecha_de_cierre,Propietario_de_Oportunidad,Fase,Hora_de_modificación,Programa_de_interés,...,VENTAS_ASESOR,Refe,Ventas Totales,Meta,LIDER,SEDE,CARGO,Nombre,Comision Unitaria,Comision Total


In [ ]:
df_novalidos

,Hora agregado,Numero de recibo,Propietario del registro,Nombre de Estudiante,Número de Documento de estudiante,Tipo de inscripción,Programa vendido,Modalidad,Periodo,Fecha de pago,...,Regional,Documento Ejecutivo_x,Número de aprobación de pago,Tipo de Financiación,Convenio,Valor pagado,Documento Ejecutivo_y,Documento Ejecutivo,Llave,MOTIVO


In [ ]:
#finalisimo

In [ ]:
with pd.ExcelWriter(
    "LIQUIDACION_VINCULACIONES_ABRIL_MAYO_26T03_REGIONALES.xlsx",
    engine="openpyxl"
) as writer:

    df_final.to_excel(
        writer,
        sheet_name="Validos",
        index=False
    )

    df_asesores_2.to_excel(
        writer,
        sheet_name="Comisiones_Asesores",
        index=False
    )

    df_novalidos.to_excel(
        writer,
        sheet_name="No_Validas",
        index=False
    )


In [ ]:
RR

NameError: name 'RR' is not defined

In [ ]:
df_final.to_excel("Liquidacion_vinculaciones_NOVIEMBRE_POS.xlsx", index=False)
df_asesores_2.to_excel("Liquidacion_comisiones_asesores_NOVIEMBRE_POS.xlsx", index=False)
df_novalidos.to_excel("Liquidacion_vinculaciones_No_validos_NOVIEMBRE_POS.xlsx", index=False)

In [ ]:
df_final['PERIODO_FINANCIERO'] = '202510'
df_final['EstadoFinal'] = 'VALIDO'
df_final['Reint_Stv'] = 0
df_final = df_final.drop(columns=['CLASE_ACTUAL','NUEVO'])
df_final = df_final.rename(columns={'PERIODO_y':'PERIODO','PROGRAMA':'NOM_PROGRAMA'})
df_final

,Documento_Estudiante,PERIODO,Nombre_de_Oportunidad,Propietario_de_Oportunidad_Name,Hora_de_creación,Fecha_de_cierre,Propietario_de_Oportunidad,Fase,Hora_de_modificación,Programa_de_interés,...,VENTAS_ASESOR,Refe,Ventas Totales,Meta,Nombre,Comision Unitaria,Comision Total,PERIODO_FINANCIERO,EstadoFinal,Reint_Stv
0,1000694004,2026A,8989-1000694004-250904090704,Vinculaciones Jeffer Sebastian Vargas Jimenez,04/09/2025 09:07,02/10/2025,4402173000785669009,Matriculado,16/10/2025 17:32,ADMINISTRACION DE EMPRESAS,...,2,0,2,40,JEFFER SEBASTIAN VARGAS JIMENEZ,0,0,202510,VALIDO,0
1,1000973546,2025D,8883-1000973546-250821064812,DANIEL FELIPE PRIETO ALVARADO,21/08/2025 18:48,21/08/2025,4402173000669519001,Matriculado,26/09/2025 12:26,INGENIERIA DE SISTEMAS,...,4,0,4,1,DANIEL FELIPE PRIETO ALVARADO,23000,92000,202510,VALIDO,0
2,1000987583,2026A,8973-1000987583-250812095636,paula andrea vence madariaga,12/08/2025 09:56,15/09/2025,4402173000808268013,Matriculado,26/09/2025 12:56,ADMINISTRACION EN SERVICIOS DE SALUD,...,2,0,2,1,PAULA ANDREA VENCE MADARIAGA,23000,46000,202510,VALIDO,0
3,1004462297,2026A,8997-1004462297-250827025501,Vinculaciones Rafael Pacheco,27/08/2025 14:55,30/08/2025,4402173000926976001,Matriculado,26/09/2025 10:12,DISEÑO GRAFICO,...,2,0,2,1,RAFAEL ALFONSO PACHECO ESCOBAR,23000,46000,202510,VALIDO,0
4,1010106273,2025D,8973-1010106273-250829050526,DANIEL FELIPE PRIETO ALVARADO,29/08/2025 17:05,08/09/2025,4402173000669519001,Matriculado,26/09/2025 10:04,ADMINISTRACION EN SERVICIOS DE SALUD,...,4,0,4,1,DANIEL FELIPE PRIETO ALVARADO,23000,92000,202510,VALIDO,0
5,1014478970,2025D,9048-1014478970-250902012226,paula andrea vence madariaga,06/09/2025 15:02,02/09/2025,4402173000808268013,Matriculado,29/09/2025 10:46,INGENIERIA DE SISTEMAS,...,2,0,2,1,PAULA ANDREA VENCE MADARIAGA,23000,46000,202510,VALIDO,0
6,1019120883,2026A,8989-1019120883-250902015322,Vinculaciones Jeffer Sebastian Vargas Jimenez,02/09/2025 13:53,01/10/2025,4402173000785669009,Matriculado,16/10/2025 17:27,ADMINISTRACION DE EMPRESAS,...,2,0,2,40,JEFFER SEBASTIAN VARGAS JIMENEZ,0,0,202510,VALIDO,0
7,1022325122,2025D,9048-1022325122-250922102049,DANIEL FELIPE PRIETO ALVARADO,22/09/2025 22:20,22/09/2025,4402173000669519001,Matriculado,29/09/2025 10:22,INGENIERIA DE SISTEMAS,...,4,0,4,1,DANIEL FELIPE PRIETO ALVARADO,23000,92000,202510,VALIDO,0
8,1030587209,25V05,8989-1030587209-250827094251,Vinculaciones Lina Paola Hernandez Palacios,27/08/2025 21:42,17/09/2025,4402173000781008015,Matriculado,26/09/2025 10:04,ADMINISTRACION DE EMPRESAS,...,1,0,1,40,LINA PAOLA HERNANDEZ PALACIOS,0,0,202510,VALIDO,0
9,1034657459,2026A,8997-1034657459-250811043747,Vinculaciones Pedro Nel Rangel Luna,11/08/2025 16:37,14/08/2025,4402173000785669007,Matriculado,26/09/2025 12:56,DISEÑO GRAFICO,...,1,0,1,40,PEDRO NEL RANGEL LUNA,0,0,202510,VALIDO,0


In [ ]:
df_final = df_final[['Documento_Estudiante', 'PERIODO', 'Nombre_de_Oportunidad',
       'Propietario_de_Oportunidad_Name', 'Hora_de_creación',
       'Fecha_de_cierre', 'Propietario_de_Oportunidad', 'Fase',
       'Hora_de_modificación', 'Programa_de_interés', 'Id',
       'Nombre_de_Interesado', 'Creado_por', 'Hora_de_la_última_actividad',
       'Correo_electrónico', 'llave1', 'Documento Ejecutivo', 'ESTADO_PAGO',
       'FuerzaComercialFinal', 'FEC_PAGO_LIQ', 'TIPO', 'NOM_PROGRAMA',
       '_merge', 'Reint_Stv', 'EstadoFinal', 'VENTAS_ASESOR', 'Refe',
       'Ventas Totales', 'Meta', 'Nombre', 'Comision Unitaria',
       'Comision Total', 'PERIODO_FINANCIERO']]

In [ ]:
# Connection to database
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
# username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
#url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
engine = create_engine(url, fast_executemany=True)

table_name = 'COMISIONES_VINCULACIONES_ACTUALES'
schema_name = 'Comisiones'

# Send the data to sql server
df_final.to_sql(
    name=table_name,
    schema=schema_name,
    con=engine,
    if_exists='append',
    index=False,
    chunksize = 10000
)

# End connection
engine.dispose()

In [ ]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 

        
        select * from comisiones.COMISIONES_VINCULACIONES_ACTUALES
 ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_historico = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [ ]:
df_historico.columns

Index(['Documento_Estudiante', 'PERIODO', 'Nombre_de_Oportunidad',
       'Propietario_de_Oportunidad_Name', 'Hora_de_creación',
       'Fecha_de_cierre', 'Propietario_de_Oportunidad', 'Fase',
       'Hora_de_modificación', 'Programa_de_interés', 'Id',
       'Nombre_de_Interesado', 'Creado_por', 'Hora_de_la_última_actividad',
       'Correo_electrónico', 'llave1', 'Documento Ejecutivo', 'ESTADO_PAGO',
       'FuerzaComercialFinal', 'FEC_PAGO_LIQ', 'TIPO', 'NOM_PROGRAMA',
       '_merge', 'Reint_Stv', 'EstadoFinal', 'VENTAS_ASESOR', 'Refe',
       'Ventas Totales', 'Meta', 'Nombre', 'Comision Unitaria',
       'Comision Total', 'PERIODO_FINANCIERO'],
      dtype='object')

In [ ]:
df = pd.read_excel('Liquidacion_vinculaciones_No_validos_AGOSTO_SEPTIEMBRE_v6.xlsx', sheet_name='Hoja2')

In [ ]:
para_orden = df['documento'].unique()
valores_unicos_df1_orden = "', '".join(map(str, para_orden))
valores_unicos_df1_orden = f"'{valores_unicos_df1_orden}'"


In [ ]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(f''' 
        SELECT distinct DOC_ALUM as Documento_Estudiante_zoho,PERIODO,ESTADO_PAGO,fuerzazoho as FuerzaComercialFinal, FEC_PAGO_LIQ, NUEVO, CLASE_ACTUAL, TIPO, PROGRAMA, est_matriculado, documento
                          FROM ZOHO.Base_Personas WITH(NOLOCK)
        WHERE  doc_alum in ({valores_unicos_df1_orden})
 ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_pos = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()
df_pos.shape

(69, 11)

In [ ]:
df['documento'] = df['documento'].astype(str)

In [ ]:
df

,documento,periodo
0,1019070872,26ET1
1,1073810835,25v05
2,1090412616,26ET1
3,10305558829,26es1
4,1082897374,26es2
5,1031146614,26es1
6,1005319598,25v05
7,1064727721,2025D
8,6000769,25v05
9,1022351813,26ES1


In [ ]:
df_pos_2 = df.merge(df_pos, how='inner', left_on='documento', right_on='Documento_Estudiante_zoho')
df_pos_2.drop_duplicates(inplace=True)
df_pos_2.shape

(69, 13)

In [ ]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(f''' 
        Select distinct 
o.Nombre_de_Oportunidad,
o.Propietario_de_Oportunidad_Name,
o.Hora_de_creación,
o.Fecha_de_cierre,
o.Propietario_de_Oportunidad,
o.Fase,
o.Hora_de_modificación,
o.Número_de_Documento,
o.Programa_de_interés,
o.Periodo,
o.Id,
o.Nombre_de_Interesado,
o.Creado_por,
o.Hora_de_la_última_actividad,
u.Correo_electrónico
from zoho.CRM.Oportunidades o
left join zoho.CRM.Usuarios U
ON o.Propietario_de_Oportunidad = u.Id
where Número_de_Documento in ({valores_unicos_df1_orden})

 ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_2 = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()
df_2.shape

(64, 15)

In [ ]:
df_2

,Nombre_de_Oportunidad,Propietario_de_Oportunidad_Name,Hora_de_creación,Fecha_de_cierre,Propietario_de_Oportunidad,Fase,Hora_de_modificación,Número_de_Documento,Programa_de_interés,Periodo,Id,Nombre_de_Interesado,Creado_por,Hora_de_la_última_actividad,Correo_electrónico
0,4852-1024575919-211111123659,Vinculaciones Regionales Cesar Parra,11/11/2021 12:36,None,4402173000085759532,Matriculado,29/11/2021 16:11,1024575919,BOGOTA CENTRO / INGENIERIA DE SISTEMAS / TECNI...,None,4402173000121730001,4402173000121702138,4402173000000251001,29/11/2021 16:11,cesar_parra@cun.edu.co
1,5147-92541481-211109123512,victor_pedraza@cun.edu.co victor_pedraza@cun.e...,09/11/2021 12:35,None,4402173000003605037,PreInscrito,19/09/2024 17:49,92541481,BOGOTA CENTRO / CONTADURIA PUBLICA / TECNICO /...,21V06,4402173000121099001,4402173000121092039,4402173000000251001,19/09/2024 17:49,asesorcundigital18@cun.edu.co
2,7026-1006117688-240620020938,Contact Luisa Maria Velandia Gomez,20/06/2024 14:09,20/06/2024,4402173000011756279,Matriculado,20/06/2024 14:13,1006117688,ESPECIALIZACION EN PAZ Y DESARROLLO TERRITORIAL,24ET3,4402173000510587027,4402173000275344403,4402173000000251001,20/06/2024 14:13,asesorcundigital67@cun.edu.co
3,7900-1070387768-240924010927,Vinculaciones NATALIA REMOLINA HERRERA,24/09/2024 13:09,27/09/2024,4402173000751702001,Matriculado,11/08/2025 11:50,1070387768,DISEO DE MODAS,2025A,4402173000553652936,4402173000551893207,4402173000000251001,11/08/2025 11:50,natalia_remolinah@cun.edu.co
4,7959-1014290554-241128053143,Contact Daniel Camilo Pinzon Pachon,28/11/2024 17:31,28/11/2024,4402173000583230011,Matriculado,29/11/2024 10:35,1014290554,DISEÑO GRAFICO,24V06,4402173000593702164,4402173000593734175,4402173000000251001,29/11/2024 10:35,daniel_pinzonp@cun.edu.co
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,9101-1031147855-250930112959,Vinculaciones NATALIA REMOLINA HERRERA,30/09/2025 11:29,01/10/2025,4402173000751702001,Matriculado,01/10/2025 23:14,1031147855,ESPECIALIZACION EN GESTION DE TECNOLOGIAS DE L...,25ET4,4402173000959465939,4402173000959465933,4402173000751702001,01/10/2025 23:14,natalia_remolinah@cun.edu.co
60,9101-80724804-250922043549,Vinculaciones NATALIA REMOLINA HERRERA,22/09/2025 16:35,23/09/2025,4402173000751702001,Matriculado,29/09/2025 12:17,80724804,ESPECIALIZACION EN GESTION DE TECNOLOGIAS DE L...,25ET4,4402173000946459974,4402173000946459968,4402173000751702001,29/09/2025 12:17,natalia_remolinah@cun.edu.co
61,9250-1012368710-251017020826,Vinculaciones NATALIA REMOLINA HERRERA,17/10/2025 14:08,None,4402173000751702001,Interes basico,17/10/2025 14:10,1012368710,INGENIERIA INDUSTRIAL,25V06,4402173000989454137,4402173000989454131,4402173000751702001,17/10/2025 14:10,natalia_remolinah@cun.edu.co
62,ESP48-1006117688-230118033310,Contact paola_pena@cun.edu.co,18/01/2023 15:33,None,4402173000003607001,Matriculado,21/01/2023 09:57,1006117688,ESPECIALIZACION EN ANALITICA DE DATOS,23ET1,4402173000275436012,4402173000275344403,4402173000000251001,14/06/2024 16:45,asesorcundigital7@cun.edu.co


In [ ]:
df_3 = df.merge(df_2, how='inner', left_on='documento', right_on='Número_de_Documento')
df_3.drop_duplicates(inplace=True)
df_3

,documento,periodo,Nombre_de_Oportunidad,Propietario_de_Oportunidad_Name,Hora_de_creación,Fecha_de_cierre,Propietario_de_Oportunidad,Fase,Hora_de_modificación,Número_de_Documento,Programa_de_interés,Periodo,Id,Nombre_de_Interesado,Creado_por,Hora_de_la_última_actividad,Correo_electrónico
0,1019070872,26ET1,8945-1019070872-250912103758,Vinculaciones NATALIA REMOLINA HERRERA,12/09/2025 10:37,19/09/2025,4402173000751702001,Matriculado,01/10/2025 08:56,1019070872,ESPECIALIZACIÓN EN DESARROLLO ORGANIZACIONAL Y...,26ES1,4402173000925966386,4402173000925966380,4402173000751702001,01/10/2025 08:56,natalia_remolinah@cun.edu.co
1,1073810835,25v05,8872-1073810835-250915025710,Vinculaciones NATALIA REMOLINA HERRERA,15/09/2025 14:57,None,4402173000751702001,Interes basico,29/09/2025 10:43,1073810835,INGENIERIA INDUSTRIAL,25V05,4402173000933224823,4402173000933224817,4402173000751702001,29/09/2025 10:43,natalia_remolinah@cun.edu.co
2,1090412616,26ET1,8945-1090412616-250912121952,Vinculaciones NATALIA REMOLINA HERRERA,12/09/2025 12:19,15/09/2025,4402173000751702001,Matriculado,01/10/2025 08:56,1090412616,ESPECIALIZACIÓN EN DESARROLLO ORGANIZACIONAL Y...,None,4402173000925966986,4402173000925966980,4402173000751702001,01/10/2025 08:56,natalia_remolinah@cun.edu.co
3,1082897374,26es2,8953-1082897374-250912060847,Vinculaciones NATALIA REMOLINA HERRERA,12/09/2025 18:08,15/09/2025,4402173000751702001,Matriculado,29/09/2025 12:17,1082897374,ESPECIALIZACIÓN EN ALTA GERENCIA,26ES1,4402173000927051145,4402173000927051139,4402173000751702001,29/09/2025 12:17,natalia_remolinah@cun.edu.co
4,1031146614,26es1,8949-1031146614-250913123747,Vinculaciones NATALIA REMOLINA HERRERA,13/09/2025 12:37,None,4402173000751702001,Interes real,29/09/2025 12:17,1031146614,ESPECIALIZACIÓN EN GERENCIA DE PROYECTOS,None,4402173000927203969,4402173000927203963,4402173000751702001,29/09/2025 12:17,natalia_remolinah@cun.edu.co
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,1121826358,26es1,8942-1121826358-251001050630,Vinculaciones NATALIA REMOLINA HERRERA,01/10/2025 17:06,01/10/2025,4402173000751702001,Matriculado,06/10/2025 16:01,1121826358,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,26ES1,4402173000962333258,4402173000962333252,4402173000751702001,06/10/2025 16:01,natalia_remolinah@cun.edu.co
60,1014227769,26es1,8944-1014227769-251001060133,Vinculaciones NATALIA REMOLINA HERRERA,01/10/2025 18:01,01/10/2025,4402173000751702001,Matriculado,06/10/2025 16:01,1014227769,ESPECIALIZACIÓN EN GERENCIA EDUCATIVA,26ES1,4402173000962362953,4402173000962362947,4402173000751702001,06/10/2025 16:01,natalia_remolinah@cun.edu.co
61,1120579442,26es1,8951-1120579442-251001061012,Vinculaciones NATALIA REMOLINA HERRERA,01/10/2025 18:10,01/10/2025,4402173000751702001,Matriculado,06/10/2025 16:00,1120579442,ESPECIALIZACIÓN EN CONTRATACIÓN ESTATAL,26ES1,4402173000962406178,4402173000962406172,4402173000751702001,06/10/2025 16:00,natalia_remolinah@cun.edu.co
62,1070387768,2025D,7900-1070387768-240924010927,Vinculaciones NATALIA REMOLINA HERRERA,24/09/2024 13:09,27/09/2024,4402173000751702001,Matriculado,11/08/2025 11:50,1070387768,DISEO DE MODAS,2025A,4402173000553652936,4402173000551893207,4402173000000251001,11/08/2025 11:50,natalia_remolinah@cun.edu.co


In [ ]:
df_pos_2.to_excel('cruce_base_personas.xlsx',index=False)

In [ ]:
df_3.to_excel('cruce_zoho.xlsx',index=False)

In [ ]:
df = pd.read_excel(r'C:\Users\alejandro_arboledas\OneDrive - Corporación Unificada Nacional de Educación Superior - CUN\Documentos\PYTHON\VINCULACIONES\MODELO3\Liquidacion_comisiones_vinculaciones_AGOSTO_SEPTIEMBRE_v2.xlsx', sheet_name='No validos')

In [ ]:
# Connection to database
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
# username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
#url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
engine = create_engine(url, fast_executemany=True)

table_name = 'VINCULACIONES_NO_VALIDOS'
schema_name = 'Comisiones'

# Send the data to sql server
df.to_sql(
    name=table_name,
    schema=schema_name,
    con=engine,
    if_exists='replace',
    index=False,
    chunksize = 10000
)

# End connection
engine.dispose()

In [ ]:
df_2 = pd.read_excel(r'C:\Users\alejandro_arboledas\OneDrive - Corporación Unificada Nacional de Educación Superior - CUN\Documentos\PYTHON\VINCULACIONES\MODELO3\Liquidacion_comisiones_asesores_AGOSTO_SEPTIEMBRE_v6.xlsx', sheet_name='Hoja2')
df_2

,Documento Ejecutivo,Nombre,Ventas Totales,Meta,Comision Unitaria,Comision Total,PERIODO_FINANCIERO
0,1026592868,NATALIA REMOLINA HERRERA,132,40,97000,12804000.0,202509
1,1018465686,ANGEL ANTONIO RIVERA PADUA,66,1,69000,4554000.0,202509
2,1128053909,MIGUEL ANGEL PALOMINO PERALTA,19,40,0,0.0,202509
3,1002152146,HUGO LEONARDO ARISTIZABAL MARTELO,13,40,0,0.0,202509
4,1110464615,CESAR AUGUSTO CANDIA MENDOZA,3,1,23000,69000.0,202509
5,1012388460,DAIRON FELIPE TELLEZ ZAMBRANO,42,40,57500,2415000.0,202509
6,1015393814,CAROLINA ROCIO VILORIA PEDREROS,39,40,0,0.0,202509
7,1033679718,LUZ DARY RAMIREZ QUEVEDO,65,40,69000,4485000.0,202509
8,1102850551,MARIA ANDREA UPARELA OVIEDO,13,40,0,0.0,202509
9,1235041897,STEFANY OSPINA TARRA,2,1,23000,46000.0,202509


In [ ]:
# Connection to database
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
# username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
#url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
engine = create_engine(url, fast_executemany=True)

table_name = 'COMISIONES_ASESOR_VINCULACIONES_ACTUALES'
schema_name = 'Comisiones'

# Send the data to sql server
df_2.to_sql(
    name=table_name,
    schema=schema_name,
    con=engine,
    if_exists='append',
    index=False,
    chunksize = 10000
)

# End connection
engine.dispose()